# geometry_with_statistics.ipynb

Emotion-centroid geometry with speaker-level statistics for CREMA-D intensity transitions.

For the final statistics section, mount the data drive and run its definitions and execution cells. Set WITHIN_PAIRS_CSV to reuse an exported pair-score table.

Use the setup steps in the repository README before running this notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =========================================================
# WEIGHTED SUBTLE CENTROIDS
# =========================================================
#
# Goal:
#
# Build annotator-weighted subtle emotion
# centroids.
#
# Example:
#
# {"joy":2, "pride":2, "approval":1}
#
# contributes:
#
# joy      -> weight 2/5
# pride    -> weight 2/5
# approval -> weight 1/5
#
# =========================================================

import ast
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

# =========================================================
# CONFIG
# =========================================================

TARGET_LAYER = "lm_2"

MODEL_FINEGRAINED = {

    "QWEN": {

        "pred_csv":
            "/content/drive/MyDrive/subprobe/"
            "task3_full_run_qwen/"
            "manipulated_probe_predictions.csv",

        "label_csv":
            "/content/drive/MyDrive/subprobe/"
            "task3_full_run_qwen/"
            "task3_results.csv"
    },

    "FLAMINGO": {

        "pred_csv":
            "/content/drive/MyDrive/subprobe/"
            "task3_full_run_flamingo/"
            "subtle_probe_predictions.csv",

        "label_csv":
            "/content/drive/MyDrive/subprobe/"
            "task3_full_run_flamingo/"
            "task3_results.csv"
    }
}

# =========================================================
# PARSERS
# =========================================================

def parse_labels(x):

    try:
        return ast.literal_eval(x)

    except:
        return []

def parse_label_counts(x):

    try:
        return ast.literal_eval(x)

    except:
        return {}

# =========================================================
# EMBEDDING LOADER
# =========================================================

def process_rep(x):

    if not torch.is_tensor(x):
        x = torch.tensor(x)

    x = x.float()

    # -----------------------------------------------------
    # pooling
    # -----------------------------------------------------

    if x.dim() == 3:
        x = x.mean(dim=1).squeeze(0)

    elif x.dim() == 2:
        x = x.squeeze(0)

    # -----------------------------------------------------
    # normalize
    # -----------------------------------------------------

    x = x / (x.norm() + 1e-8)

    return x.numpy()

def load_embedding(path, layer):

    try:

        rep = torch.load(
            path,
            map_location="cpu"
        )

        if layer not in rep:
            return None

        return process_rep(
            rep[layer]
        )

    except:
        return None

# =========================================================
# BUILD WEIGHTED CENTROIDS
# =========================================================

model_centroids = {}

for model_name, cfg in MODEL_FINEGRAINED.items():

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    # =====================================================
    # LOAD
    # =====================================================

    pred_df = pd.read_csv(
        cfg["pred_csv"]
    )

    label_df = pd.read_csv(
        cfg["label_csv"]
    )

    # -----------------------------------------------------
    # clean paths
    # -----------------------------------------------------

    pred_df["rep_path"] = (
        pred_df["rep_path"]
        .astype(str)
        .str.strip()
    )

    label_df["rep_path"] = (
        label_df["rep_path"]
        .astype(str)
        .str.strip()
    )

    # =====================================================
    # MERGE
    # =====================================================

    df = pd.merge(

        pred_df,

        label_df[
            [
                "rep_path",
                "labels",
                "label_counts"
            ]
        ],

        on="rep_path",

        how="inner"
    )

    # =====================================================
    # FILTER LAYER
    # =====================================================

    df = df[
        df["layer"] == TARGET_LAYER
    ].copy()

    print("Merged rows:", len(df))

    # =====================================================
    # PARSE LABELS
    # =====================================================

    df["subtle_labels"] = (
        df["labels"]
        .apply(parse_labels)
    )

    df["label_count_dict"] = (
        df["label_counts"]
        .apply(parse_label_counts)
    )

    # =====================================================
    # COLLECT ALL SUBTLE LABELS
    # =====================================================

    all_subtle = set()

    for d in df["label_count_dict"]:

        for k in d.keys():
            all_subtle.add(k)

    print(
        "Unique subtle labels:",
        len(all_subtle)
    )

    # =====================================================
    # BUILD CENTROIDS
    # =====================================================

    centroids = {}

    for subtle in sorted(all_subtle):

        weighted_embs = []

        weights = []

        # -------------------------------------------------
        # iterate samples
        # -------------------------------------------------

        for _, row in tqdm(
            df.iterrows(),
            total=len(df),
            desc=subtle
        ):

            count_dict = row[
                "label_count_dict"
            ]

            # ---------------------------------------------
            # skip if subtle absent
            # ---------------------------------------------

            if subtle not in count_dict:
                continue

            # ---------------------------------------------
            # weight
            # ---------------------------------------------

            total_votes = sum(
                count_dict.values()
            )

            if total_votes == 0:
                continue

            weight = (
                count_dict[subtle]
                /
                total_votes
            )

            # ---------------------------------------------
            # embedding
            # ---------------------------------------------

            emb = load_embedding(
                row["rep_path"],
                TARGET_LAYER
            )

            if emb is None:
                continue

            weighted_embs.append(
                emb * weight
            )

            weights.append(weight)

        # -------------------------------------------------
        # finalize centroid
        # -------------------------------------------------

        if len(weighted_embs) == 0:
            continue

        centroid = (
            np.sum(
                weighted_embs,
                axis=0
            )
            /
            (
                np.sum(weights)
                + 1e-8
            )
        )

        # normalize
        centroid = (
            centroid
            /
            (
                np.linalg.norm(centroid)
                + 1e-8
            )
        )

        centroids[subtle] = centroid

        print(
            f"{subtle}: "
            f"{len(weights)} samples"
        )

    # =====================================================
    # SAVE MODEL CENTROIDS
    # =====================================================

    model_centroids[
        model_name
    ] = centroids

    print(
        f"\nBuilt {len(centroids)} centroids"
    )

In [ ]:
# =========================================================
# BLOCK 2
# BUILD CANONICAL COARSE CENTROIDS
# =========================================================
#
# Goal:
#
# Create CLEAN coarse centroids using
# only canonical/basic emotion samples.
#
# Why:
#
# subtle direction =
#   subtle centroid - coarse centroid
#
# becomes much cleaner if coarse centroids
# are NOT contaminated by subtle mixtures.
#
# =========================================================

import ast
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

# =========================================================
# BASIC EMOTIONS
# =========================================================

BASIC_MAP = {

    "anger": "ANG",

    "disgust": "DIS",

    "fear": "FEA",

    "joy": "HAP",

    "sadness": "SAD",

    "surprise": "SUR",

    "neutral": "NEU"
}

# =========================================================
# CONFIG
# =========================================================

TARGET_LAYER = "lm_2"

MODEL_FINEGRAINED = {

    "QWEN": {

        "pred_csv":
            "/content/drive/MyDrive/subprobe/"
            "task3_full_run_qwen/"
            "manipulated_probe_predictions.csv",

        "label_csv":
            "/content/drive/MyDrive/subprobe/"
            "task3_full_run_qwen/"
            "task3_results.csv"
    },

    "FLAMINGO": {

        "pred_csv":
            "/content/drive/MyDrive/subprobe/"
            "task3_full_run_flamingo/"
            "subtle_probe_predictions.csv",

        "label_csv":
            "/content/drive/MyDrive/subprobe/"
            "task3_full_run_flamingo/"
            "task3_results.csv"
    }
}

# =========================================================
# PARSERS
# =========================================================

def parse_labels(x):

    try:
        return ast.literal_eval(x)

    except:
        return []

def parse_label_counts(x):

    try:
        return ast.literal_eval(x)

    except:
        return {}

# =========================================================
# EMBEDDING
# =========================================================

def process_rep(x):

    if not torch.is_tensor(x):
        x = torch.tensor(x)

    x = x.float()

    # -----------------------------------------------------
    # pooling
    # -----------------------------------------------------

    if x.dim() == 3:
        x = x.mean(dim=1).squeeze(0)

    elif x.dim() == 2:
        x = x.squeeze(0)

    # -----------------------------------------------------
    # normalize
    # -----------------------------------------------------

    x = x / (x.norm() + 1e-8)

    return x.numpy()

def load_embedding(path, layer):

    try:

        rep = torch.load(
            path,
            map_location="cpu"
        )

        if layer not in rep:
            return None

        return process_rep(
            rep[layer]
        )

    except:
        return None

# =========================================================
# BUILD COARSE CENTROIDS
# =========================================================

model_coarse_centroids = {}

for model_name, cfg in MODEL_FINEGRAINED.items():

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    # =====================================================
    # LOAD
    # =====================================================

    pred_df = pd.read_csv(
        cfg["pred_csv"]
    )

    label_df = pd.read_csv(
        cfg["label_csv"]
    )

    pred_df["rep_path"] = (
        pred_df["rep_path"]
        .astype(str)
        .str.strip()
    )

    label_df["rep_path"] = (
        label_df["rep_path"]
        .astype(str)
        .str.strip()
    )

    # =====================================================
    # MERGE
    # =====================================================

    df = pd.merge(

        pred_df,

        label_df[
            [
                "rep_path",
                "labels",
                "label_counts"
            ]
        ],

        on="rep_path",

        how="inner"
    )

    # =====================================================
    # FILTER LAYER
    # =====================================================

    df = df[
        df["layer"] == TARGET_LAYER
    ].copy()

    # =====================================================
    # PARSE
    # =====================================================

    df["subtle_labels"] = (
        df["labels"]
        .apply(parse_labels)
    )

    df["label_count_dict"] = (
        df["label_counts"]
        .apply(parse_label_counts)
    )

    # =====================================================
    # BUILD BASIC CENTROIDS
    # =====================================================

    coarse_centroids = {}

    for subtle_basic, coarse_name in BASIC_MAP.items():

        embs = []

        weights = []

        # -------------------------------------------------
        # iterate samples
        # -------------------------------------------------

        for _, row in tqdm(
            df.iterrows(),
            total=len(df),
            desc=coarse_name
        ):

            count_dict = row[
                "label_count_dict"
            ]

            # ---------------------------------------------
            # must contain target basic emotion
            # ---------------------------------------------

            if subtle_basic not in count_dict:
                continue

            # ---------------------------------------------
            # dominance filtering
            # keep only samples where canonical
            # emotion is strongest
            # ---------------------------------------------

            max_count = max(
                count_dict.values()
            )

            if (
                count_dict[subtle_basic]
                < max_count
            ):
                continue

            # ---------------------------------------------
            # compute weight
            # ---------------------------------------------

            total_votes = sum(
                count_dict.values()
            )

            weight = (
                count_dict[subtle_basic]
                /
                total_votes
            )

            # ---------------------------------------------
            # embedding
            # ---------------------------------------------

            emb = load_embedding(
                row["rep_path"],
                TARGET_LAYER
            )

            if emb is None:
                continue

            embs.append(
                emb * weight
            )

            weights.append(weight)

        # -------------------------------------------------
        # finalize centroid
        # -------------------------------------------------

        if len(embs) == 0:
            continue

        centroid = (

            np.sum(
                embs,
                axis=0
            )

            /

            (
                np.sum(weights)
                + 1e-8
            )
        )

        centroid = (
            centroid
            /
            (
                np.linalg.norm(centroid)
                + 1e-8
            )
        )

        coarse_centroids[
            coarse_name
        ] = centroid

        print(
            f"{coarse_name}: "
            f"{len(weights)} samples"
        )

    # =====================================================
    # SAVE
    # =====================================================

    model_coarse_centroids[
        model_name
    ] = coarse_centroids

    print(
        f"\nBuilt "
        f"{len(coarse_centroids)} "
        f"coarse centroids"
    )

In [ ]:
# =========================================================
# BLOCK 3
# LOAD INTENSITY TRANSITIONS
# =========================================================
#
# Goal:
#
# Build clean transition dataframe:
#
# MD -> HI
# LO -> MD
# LO -> HI
#
# using:
#
# source_rep = neutral_rep_path
# target_rep = rep_path
#
# =========================================================

import pandas as pd

# =========================================================
# CONFIG
# =========================================================

MODEL_TRANSITIONS = {

    "QWEN": {

        "csv":
            "/content/drive/MyDrive/probing/"
            "full_probe/albe/"
            "merged_results.csv"
    },

    "FLAMINGO": {

        "csv":
            "/content/drive/MyDrive/probing/"
            "full_probe_flamingo/abla/"
            "merged_results.csv"
    }
}

# =========================================================
# PARSER
# =========================================================

def parse_intensity_path(path):

    """
    Example:
    subset_audio\\1023_IEO_ANG_HI.wav
    """

    try:

        # ---------------------------------------------
        # windows path safe
        # ---------------------------------------------

        fname = str(path).split("\\")[-1]

        fname = fname.replace(
            ".wav",
            ""
        )

        parts = fname.split("_")

        return {

            "speaker":
                parts[0],

            "utterance":
                parts[1],

            "coarse":
                parts[2],

            "intensity":
                parts[3]
        }

    except Exception as e:

        print("Parse fail:", path)

        return None

# =========================================================
# BUILD TRANSITIONS
# =========================================================

all_rows = []

for model_name, cfg in MODEL_TRANSITIONS.items():

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    df = pd.read_csv(
        cfg["csv"]
    )

    # =====================================================
    # INTENSITY ONLY
    # =====================================================

    df = df[
        df["task"] == "intensity"
    ].copy()

    print("Rows:", len(df))

    # =====================================================
    # BUILD TRANSITIONS
    # =====================================================

    for _, row in df.iterrows():

        src = parse_intensity_path(
            row["neutral_path"]
        )

        tgt = parse_intensity_path(
            row["manipulated_path"]
        )

        if src is None or tgt is None:
            continue

        # -------------------------------------------------
        # sanity checks
        # -------------------------------------------------

        if (
            src["speaker"]
            !=
            tgt["speaker"]
        ):
            continue

        if (
            src["utterance"]
            !=
            tgt["utterance"]
        ):
            continue

        if (
            src["coarse"]
            !=
            tgt["coarse"]
        ):
            continue

        # =================================================
        # SAVE
        # =================================================

        all_rows.append({

            # ---------------------------------------------
            # model
            # ---------------------------------------------

            "model":
                model_name,

            # ---------------------------------------------
            # metadata
            # ---------------------------------------------

            "speaker":
                src["speaker"],

            "utterance":
                src["utterance"],

            "coarse":
                src["coarse"],

            # ---------------------------------------------
            # transition
            # ---------------------------------------------

            "source_intensity":
                src["intensity"],

            "target_intensity":
                tgt["intensity"],

            "transition":
                (
                    f"{src['intensity']}"
                    f"_TO_"
                    f"{tgt['intensity']}"
                ),

            # ---------------------------------------------
            # representations
            # ---------------------------------------------

            "source_rep":
                row["neutral_rep_path"],

            "target_rep":
                row["rep_path"],

            # ---------------------------------------------
            # labels
            # ---------------------------------------------

            "source_label":
                row.get(
                    "neutral_label",
                    None
                ),

            "target_label":
                row.get(
                    "label",
                    None
                )
        })

# =========================================================
# FINAL DF
# =========================================================

transition_df = pd.DataFrame(all_rows)

print("\nFinal transitions:")
print(len(transition_df))

# =========================================================
# INSPECT
# =========================================================

print("\nTransition counts:\n")

print(
    transition_df["transition"]
    .value_counts()
)

print("\nCoarse counts:\n")

print(
    transition_df["coarse"]
    .value_counts()
)

print("\nExample:\n")

print(
    transition_df.head(10)
)

In [ ]:
# =========================================================
# BLOCK 4
# DIRECTIONAL FINE-GRAINED SHIFT ANALYSIS
# =========================================================
#
# Core idea:
#
# displacement =
#   target_embedding - source_embedding
#
# compare against:
#
# subtle_direction =
#   subtle_centroid - coarse_centroid
#
# using cosine similarity.
#
# =========================================================

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

# =========================================================
# COARSE -> SUBTLE MAP
# =========================================================

TARGET_SUBTLE = {

    # -----------------------------------------------------
    # ANGER
    # -----------------------------------------------------

    "ANG": [

        "anger",

        "annoyance",

        "disapproval"
    ],

    # -----------------------------------------------------
    # DISGUST
    # -----------------------------------------------------

    "DIS": [

        "disgust",

        "disapproval"
    ],

    # -----------------------------------------------------
    # FEAR
    # -----------------------------------------------------

    "FEA": [

        "fear",

        "nervousness"
    ],

    # -----------------------------------------------------
    # HAPPINESS
    # -----------------------------------------------------

    "HAP": [

        "joy",

        "amusement",

        "excitement",

        "pride",

        "optimism",

        "love",

        "admiration",

        "relief",

        "gratitude",

        "caring",

        "desire"
    ],

    # -----------------------------------------------------
    # NEUTRAL
    # -----------------------------------------------------

    "NEU": [

        "confusion",

        "curiosity",

        "realization",

        "surprise"
    ],

    # -----------------------------------------------------
    # SADNESS
    # -----------------------------------------------------

    "SAD": [

        "sadness",

        "disappointment",

        "remorse",

        "grief"
    ],

    # -----------------------------------------------------
    # SURPRISE
    # -----------------------------------------------------

    "SUR": [

        "surprise",

        "realization"
    ]
}

# =========================================================
# RUN ANALYSIS
# =========================================================

results = []

for model_name in transition_df["model"].unique():

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    # =====================================================
    # MODEL DATA
    # =====================================================

    model_df = transition_df[
        transition_df["model"]
        == model_name
    ]

    # -----------------------------------------------------
    # model-specific centroids
    # -----------------------------------------------------

    subtle_centroids = (
        model_centroids[model_name]
    )

    coarse_centroids = (
        model_coarse_centroids[model_name]
    )

    # =====================================================
    # TRANSITIONS
    # =====================================================

    for _, row in tqdm(
        model_df.iterrows(),
        total=len(model_df)
    ):

        coarse = row["coarse"]

        # -------------------------------------------------
        # skip missing
        # -------------------------------------------------

        if coarse not in TARGET_SUBTLE:
            continue

        if coarse not in coarse_centroids:
            continue

        # =================================================
        # LOAD EMBEDDINGS
        # =================================================

        src_emb = load_embedding(
            row["source_rep"],
            TARGET_LAYER
        )

        tgt_emb = load_embedding(
            row["target_rep"],
            TARGET_LAYER
        )

        if src_emb is None or tgt_emb is None:
            continue

        # =================================================
        # REPRESENTATION DISPLACEMENT
        # =================================================

        delta = tgt_emb - src_emb

        delta_norm = np.linalg.norm(
            delta
        )

        if delta_norm < 1e-8:
            continue

        delta = (
            delta
            /
            delta_norm
        )

        # =================================================
        # SUBTLE DIRECTION ALIGNMENT
        # =================================================

        coarse_centroid = (
            coarse_centroids[coarse]
        )

        for subtle in TARGET_SUBTLE[coarse]:

            # ---------------------------------------------
            # subtle centroid exists?
            # ---------------------------------------------

            if subtle not in subtle_centroids:
                continue

            subtle_centroid = (
                subtle_centroids[subtle]
            )

            # ---------------------------------------------
            # direction:
            #
            # coarse -> subtle
            # ---------------------------------------------

            subtle_direction = (

                subtle_centroid
                -
                coarse_centroid
            )

            subtle_norm = np.linalg.norm(
                subtle_direction
            )

            if subtle_norm < 1e-8:
                continue

            subtle_direction = (

                subtle_direction
                /
                subtle_norm
            )

            # ---------------------------------------------
            # cosine alignment
            # ---------------------------------------------

            sim = cosine_similarity(

                delta.reshape(1, -1),

                subtle_direction.reshape(1, -1)

            )[0][0]

            # =================================================
            # SAVE
            # =================================================

            results.append({

                # ---------------------------------------------
                # model
                # ---------------------------------------------

                "model":
                    model_name,

                # ---------------------------------------------
                # metadata
                # ---------------------------------------------

                "speaker":
                    row["speaker"],

                "utterance":
                    row["utterance"],

                "coarse":
                    coarse,

                # ---------------------------------------------
                # transition
                # ---------------------------------------------

                "transition":
                    row["transition"],

                "source_intensity":
                    row["source_intensity"],

                "target_intensity":
                    row["target_intensity"],

                # ---------------------------------------------
                # subtle direction
                # ---------------------------------------------

                "subtle":
                    subtle,

                # ---------------------------------------------
                # metric
                # ---------------------------------------------

                "directional_similarity":
                    sim
            })

# =========================================================
# FINAL RESULTS
# =========================================================

shift_df = pd.DataFrame(results)

print("\nTotal results:")
print(len(shift_df))

# =========================================================
# SUMMARY
# =========================================================

summary = (

    shift_df

    .groupby([
        "model",
        "coarse",
        "transition",
        "subtle"
    ])

    ["directional_similarity"]

    .agg([
        "mean",
        "std",
        "count"
    ])

    .reset_index()

)

print("\nSummary:\n")

print(summary.head(100))

In [ ]:
# =========================================================
# BLOCK 5
# CANONICAL vs SUBTLE DIRECTION COMPARISON
# =========================================================
#
# Goal:
#
# Compare whether intensity shifts move toward:
#
# (1) canonical/basic emotion direction
# (2) subtle emotion direction
#
# Hypothesis:
#
# MD -> HI
#   aligns more with canonical direction
#
# MD -> LO
#   aligns more with subtle directions
#
# =========================================================

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

# =========================================================
# CANONICAL LABEL MAP
# =========================================================

CANONICAL_LABEL = {

    "ANG": "anger",

    "DIS": "disgust",

    "FEA": "fear",

    "HAP": "joy",

    "SAD": "sadness",

    "SUR": "surprise",

    "NEU": "neutral"
}

# =========================================================
# SUBTLE MAP
# =========================================================

TARGET_SUBTLE = {

    "ANG": [
        "annoyance",
        "disapproval"
    ],

    "DIS": [
        "disapproval"
    ],

    "FEA": [
        "nervousness"
    ],

    "HAP": [
        "amusement",
        "excitement",
        "pride",
        "optimism",
        "love",
        "admiration",
        "relief",
        "gratitude",
        "caring",
        "desire"
    ],

    "NEU": [
        "confusion",
        "curiosity",
        "realization"
    ],

    "SAD": [
        "disappointment",
        "remorse",
        "grief"
    ],

    "SUR": [
        "realization"
    ]
}

# =========================================================
# ANALYSIS
# =========================================================

comparison_results = []

for model_name in transition_df["model"].unique():

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    model_df = transition_df[
        transition_df["model"]
        == model_name
    ]

    subtle_centroids = (
        model_centroids[model_name]
    )

    coarse_centroids = (
        model_coarse_centroids[model_name]
    )

    # =====================================================
    # TRANSITIONS
    # =====================================================

    for _, row in tqdm(
        model_df.iterrows(),
        total=len(model_df)
    ):

        coarse = row["coarse"]

        if coarse not in coarse_centroids:
            continue

        # -------------------------------------------------
        # canonical label
        # -------------------------------------------------

        canonical = CANONICAL_LABEL.get(
            coarse,
            None
        )

        if canonical is None:
            continue

        if canonical not in subtle_centroids:
            continue

        # =================================================
        # LOAD EMBEDDINGS
        # =================================================

        src_emb = load_embedding(
            row["source_rep"],
            TARGET_LAYER
        )

        tgt_emb = load_embedding(
            row["target_rep"],
            TARGET_LAYER
        )

        if src_emb is None or tgt_emb is None:
            continue

        # =================================================
        # DISPLACEMENT
        # =================================================

        delta = tgt_emb - src_emb

        delta_norm = np.linalg.norm(delta)

        if delta_norm < 1e-8:
            continue

        delta = delta / delta_norm

        # =================================================
        # CANONICAL DIRECTION
        # =================================================

        coarse_centroid = (
            coarse_centroids[coarse]
        )

        canonical_centroid = (
            subtle_centroids[canonical]
        )

        canonical_direction = (

            canonical_centroid
            -
            coarse_centroid
        )

        canonical_norm = np.linalg.norm(
            canonical_direction
        )

        if canonical_norm > 1e-8:

            canonical_direction = (

                canonical_direction
                /
                canonical_norm
            )

            canonical_sim = cosine_similarity(

                delta.reshape(1, -1),

                canonical_direction.reshape(1, -1)

            )[0][0]

        else:

            canonical_sim = np.nan

        # =================================================
        # SUBTLE ALIGNMENT
        # =================================================

        subtle_scores = []

        for subtle in TARGET_SUBTLE.get(
            coarse,
            []
        ):

            if subtle not in subtle_centroids:
                continue

            subtle_direction = (

                subtle_centroids[subtle]
                -
                coarse_centroid
            )

            subtle_norm = np.linalg.norm(
                subtle_direction
            )

            if subtle_norm < 1e-8:
                continue

            subtle_direction = (

                subtle_direction
                /
                subtle_norm
            )

            sim = cosine_similarity(

                delta.reshape(1, -1),

                subtle_direction.reshape(1, -1)

            )[0][0]

            subtle_scores.append(sim)

        # -------------------------------------------------
        # average subtle alignment
        # -------------------------------------------------

        if len(subtle_scores) == 0:
            continue

        avg_subtle_sim = np.mean(
            subtle_scores
        )

        # =================================================
        # SAVE
        # =================================================

        comparison_results.append({

            "model":
                model_name,

            "coarse":
                coarse,

            "transition":
                row["transition"],

            # ---------------------------------------------
            # scores
            # ---------------------------------------------

            "canonical_alignment":
                canonical_sim,

            "subtle_alignment":
                avg_subtle_sim,

            # ---------------------------------------------
            # difference
            # positive:
            # more canonical
            #
            # negative:
            # more subtle
            # ---------------------------------------------

            "canonical_minus_subtle":
                (
                    canonical_sim
                    -
                    avg_subtle_sim
                )
        })

# =========================================================
# FINAL DF
# =========================================================

comparison_df = pd.DataFrame(
    comparison_results
)

print("\nTotal rows:")
print(len(comparison_df))

# =========================================================
# SUMMARY
# =========================================================

summary = (

    comparison_df

    .groupby([
        "model",
        "coarse",
        "transition"
    ])

    [
        [
            "canonical_alignment",
            "subtle_alignment",
            "canonical_minus_subtle"
        ]
    ]

    .mean()

    .reset_index()

)

print("\nSummary:\n")

print(summary)

In [ ]:
shift_df.to_csv(
    "fine_grained_shift_results.csv",
    index=False
)

In [ ]:
summary.to_csv(
    "fine_grained_shift_summary.csv",
    index=False
)

In [ ]:
comparison_df.to_csv(
    "canonical_subtle_comparison.csv",
    index=False
)

In [ ]:
comparison_summary = (
    comparison_df
    .groupby([
        "model",
        "coarse",
        "transition"
    ])
    [[
        "canonical_alignment",
        "subtle_alignment",
        "canonical_minus_subtle"
    ]]
    .agg([
        "mean",
        "std"
    ])
    .reset_index()
)

comparison_summary.to_csv(
    "canonical_subtle_summary.csv",
    index=False
)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plot_df = (
    comparison_df
    .groupby([
        "model",
        "transition"
    ])
    ["subtle_alignment"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7,5))

for model in plot_df["model"].unique():

    sub = plot_df[
        plot_df["model"] == model
    ]

    ax.plot(
        sub["transition"],
        sub["subtle_alignment"],
        marker="o",
        label=model
    )

ax.axhline(0, linestyle="--")

ax.set_ylabel(
    "Mean subtle-direction alignment"
)

ax.set_xlabel(
    "Intensity transition"
)

ax.set_title(
    "Speech intensity perturbation and subtle-emotion alignment"
)

ax.legend()

plt.tight_layout()

plt.savefig(
    "subtle_alignment_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =========================================================
# AGGREGATE
# =========================================================

plot_df = (
    comparison_df
    .groupby([
        "model",
        "coarse",
        "transition"
    ])
    ["subtle_alignment"]
    .mean()
    .reset_index()
)

coarse_list = sorted(
    plot_df["coarse"].unique()
)

# =========================================================
# SIDE-BY-SIDE FIGURE
# =========================================================

models = plot_df["model"].unique()

fig, axes = plt.subplots(
    1,
    len(models),
    figsize=(14, 5),
    sharey=True
)

# ---------------------------------------------------------
# handle single model case
# ---------------------------------------------------------

if len(models) == 1:
    axes = [axes]

# =========================================================
# PLOT EACH MODEL
# =========================================================

for ax, model in zip(axes, models):

    sub = plot_df[
        plot_df["model"] == model
    ]

    x = np.arange(len(coarse_list))

    width = 0.35

    vals_hi = []
    vals_lo = []

    for coarse in coarse_list:

        hi = sub[
            (sub["coarse"] == coarse)
            &
            (sub["transition"] == "MD_TO_HI")
        ]["subtle_alignment"]

        lo = sub[
            (sub["coarse"] == coarse)
            &
            (sub["transition"] == "MD_TO_LO")
        ]["subtle_alignment"]

        vals_hi.append(
            hi.iloc[0] if len(hi) else 0
        )

        vals_lo.append(
            lo.iloc[0] if len(lo) else 0
        )

    # -----------------------------------------------------
    # bars
    # -----------------------------------------------------

    ax.bar(
        x - width/2,
        vals_hi,
        width,
        label="MD→HI"
    )

    ax.bar(
        x + width/2,
        vals_lo,
        width,
        label="MD→LO"
    )

    # -----------------------------------------------------
    # formatting
    # -----------------------------------------------------

    ax.axhline(
        0,
        linestyle="--"
    )

    ax.set_xticks(x)

    ax.set_xticklabels(coarse_list)

    ax.set_title(model)

    ax.set_xlabel(
        "Coarse emotion"
    )

# =========================================================
# SHARED LABELS
# =========================================================

axes[0].set_ylabel(
    "Subtle-direction alignment"
)

# =========================================================
# SHARED LEGEND
# =========================================================

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=2,
    bbox_to_anchor=(0.5, 1.02)
)

# =========================================================
# TITLE
# =========================================================

fig.suptitle(
    "Intensity-induced representational shift toward subtle emotional regions",
    y=1.08
)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    "combined_coarse_shift_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =========================================================
# HEATMAP:
# WHICH SUBTLE EMOTION DOES THE SHIFT MOVE TOWARD?
# =========================================================

import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# AGGREGATE
# =========================================================

heatmap_df = (

    shift_df

    .groupby([
        "model",
        "transition",
        "coarse",
        "subtle"
    ])

    ["directional_similarity"]

    .mean()

    .reset_index()
)

print(heatmap_df.head())

# =========================================================
# PLOT
# =========================================================

for model_name in heatmap_df["model"].unique():

    for transition in heatmap_df["transition"].unique():

        sub = heatmap_df[

            (heatmap_df["model"] == model_name)
            &
            (heatmap_df["transition"] == transition)

        ]

        # -------------------------------------------------
        # pivot table
        # rows = coarse
        # cols = subtle
        # -------------------------------------------------

        pivot = sub.pivot(

            index="coarse",
            columns="subtle",
            values="directional_similarity"
        )

        # -------------------------------------------------
        # plot
        # -------------------------------------------------

        fig, ax = plt.subplots(
            figsize=(12, 4)
        )

        im = ax.imshow(
            pivot.values,
            aspect="auto"
        )

        # -------------------------------------------------
        # ticks
        # -------------------------------------------------

        ax.set_xticks(
            range(len(pivot.columns))
        )

        ax.set_xticklabels(
            pivot.columns,
            rotation=45,
            ha="right"
        )

        ax.set_yticks(
            range(len(pivot.index))
        )

        ax.set_yticklabels(
            pivot.index
        )

        # -------------------------------------------------
        # values
        # -------------------------------------------------

        for i in range(pivot.shape[0]):

            for j in range(pivot.shape[1]):

                val = pivot.values[i, j]

                if pd.isna(val):
                    continue

                ax.text(
                    j,
                    i,
                    f"{val:.2f}",
                    ha="center",
                    va="center",
                    fontsize=8
                )

        # -------------------------------------------------
        # title
        # -------------------------------------------------

        ax.set_title(
            f"{model_name} | {transition}\n"
            "Subtle-emotion directional alignment"
        )

        # -------------------------------------------------
        # colorbar
        # -------------------------------------------------

        cbar = plt.colorbar(im)

        cbar.set_label(
            "Directional similarity"
        )

        plt.tight_layout()

        # -------------------------------------------------
        # save
        # -------------------------------------------------

        plt.savefig(
            f"{model_name}_{transition}_heatmap.png",
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()

In [ ]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

TARGET_LAYER = "lm_2"

MODELS = {
    "QWEN": {
        "csv": "/content/drive/MyDrive/probe/probe_representations_2/probe_rep_index.csv"
    },
    "FLAMINGO": {
        "csv": "/content/drive/MyDrive/probe/probe_representations_flamingo_2/probe_rep_index.csv"
    }
}

def process_rep(x):

    if not torch.is_tensor(x):
        x = torch.tensor(x)

    x = x.float()

    if x.dim() == 3:
        x = x.mean(dim=1).squeeze(0)
    elif x.dim() == 2:
        x = x.squeeze(0)

    x = x / (x.norm() + 1e-8)

    return x.numpy()


def load_embedding(path, layer):

    rep = torch.load(path, map_location="cpu")

    if layer not in rep:
        return None

    return process_rep(rep[layer])


model_centroids = {}

for model_name, cfg in MODELS.items():

    print("="*60)
    print(model_name)
    print("="*60)

    df = pd.read_csv(cfg["csv"])

    df["rep_path"] = df["rep_path"].astype(str).str.strip()

    centroids = {}

    for emotion in sorted(df["emotion"].unique()):

        subset = df[df["emotion"] == emotion]

        embs = []

        for _, row in tqdm(subset.iterrows(),
                           total=len(subset),
                           desc=emotion):

            emb = load_embedding(
                row["rep_path"],
                TARGET_LAYER
            )

            if emb is not None:
                embs.append(emb)

        embs = np.stack(embs)

        centroid = embs.mean(axis=0)
        centroid /= np.linalg.norm(centroid) + 1e-8

        centroids[emotion] = centroid

        print(f"{emotion}: {len(embs)} samples")

    model_centroids[model_name] = centroids

In [ ]:
# =========================================================
# BLOCK 5 (v2)
# CENTROID DISTANCE ANALYSIS
# =========================================================
#
# Question:
#
# Does increasing intensity move representations
# closer to the canonical emotion centroid?
#
# MD -> HI : should become closer
# MD -> LO : should become further
#
# =========================================================

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

comparison_results = []

for model_name in transition_df["model"].unique():

    print("\n" + "="*60)
    print(model_name)
    print("="*60)

    model_df = transition_df[
        transition_df["model"] == model_name
    ]

    centroids = model_centroids[model_name]

    for _, row in tqdm(
        model_df.iterrows(),
        total=len(model_df)
    ):

        emotion = row["coarse"]

        if emotion not in centroids:
            raise ValueError(f"Missing centroid: {model_name}/{emotion}")

        centroid = centroids[emotion]

        # ----------------------------------------
        # load embeddings
        # ----------------------------------------

        src_emb = load_embedding(
            row["source_rep"],
            TARGET_LAYER
        )

        tgt_emb = load_embedding(
            row["target_rep"],
            TARGET_LAYER
        )

        if src_emb is None or tgt_emb is None:
            raise ValueError(f"Missing {TARGET_LAYER} embedding for {row.to_dict()}")

        # ----------------------------------------
        # cosine similarity to centroid
        # ----------------------------------------

        src_sim = cosine_similarity(
            src_emb.reshape(1,-1),
            centroid.reshape(1,-1)
        )[0][0]

        tgt_sim = cosine_similarity(
            tgt_emb.reshape(1,-1),
            centroid.reshape(1,-1)
        )[0][0]

        comparison_results.append({

            "speaker": str(row["speaker"]),
            "utterance": str(row["utterance"]),
            "layer": TARGET_LAYER,
            "source_rep": row["source_rep"],
            "target_rep": row["target_rep"],

            "model": model_name,

            "emotion": emotion,

            "transition": row["transition"],

            "source_similarity": src_sim,

            "target_similarity": tgt_sim,

            "similarity_change":
                tgt_sim - src_sim

        })

# =========================================================
# FINAL DATAFRAME
# =========================================================

comparison_df = pd.DataFrame(
    comparison_results
)

print("Total rows:", len(comparison_df))

# =========================================================
# SUMMARY
# =========================================================

summary = (

    comparison_df

    .groupby([
        "model",
        "transition"
    ])

    [
        [
            "source_similarity",
            "target_similarity",
            "similarity_change"
        ]
    ]

    .mean()

    .reset_index()

)

print(summary)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =========================================================
# AGGREGATE
# =========================================================

plot_df = (
    comparison_df
    .groupby([
        "model",
        "emotion",
        "transition"
    ])["similarity_change"]
    .mean()
    .reset_index()
)

emotion_list = sorted(plot_df["emotion"].unique())

# =========================================================
# FIGURE
# =========================================================

models = plot_df["model"].unique()

fig, axes = plt.subplots(
    1,
    len(models),
    figsize=(14,5),
    sharey=False
)

if len(models) == 1:
    axes = [axes]

# =========================================================
# PLOT
# =========================================================

for ax, model in zip(axes, models):

    sub = plot_df[
        plot_df["model"] == model
    ]

    x = np.arange(len(emotion_list))
    width = 0.35

    vals_hi = []
    vals_lo = []

    for emotion in emotion_list:

        hi = sub[
            (sub["emotion"] == emotion)
            &
            (sub["transition"] == "MD_TO_HI")
        ]["similarity_change"]

        lo = sub[
            (sub["emotion"] == emotion)
            &
            (sub["transition"] == "MD_TO_LO")
        ]["similarity_change"]

        vals_hi.append(
            hi.iloc[0] if len(hi) else np.nan
        )

        vals_lo.append(
            lo.iloc[0] if len(lo) else np.nan
        )

    ax.bar(
        x - width/2,
        vals_hi,
        width,
        label="MD→HI"
    )

    ax.bar(
        x + width/2,
        vals_lo,
        width,
        label="MD→LO"
    )

    ax.axhline(
        0,
        linestyle="--"
    )

    ax.set_xticks(x)
    ax.set_xticklabels(emotion_list)

    ax.set_title(model)

    ax.set_xlabel(
        "Emotion"
    )

# =========================================================
# LABELS
# =========================================================

axes[0].set_ylabel(
    "Change in cosine similarity\n(to emotion centroid)"
)

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=2,
    bbox_to_anchor=(0.5,1.02)
)

fig.suptitle(
    "Effect of intensity manipulation on similarity to emotion centroids",
    y=1.08
)

plt.tight_layout()

plt.savefig(
    "centroid_similarity_shift.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:


table = (
    comparison_df
    .groupby([
        "model",
        "emotion",
        "transition"
    ])["similarity_change"]
    .mean()
    .unstack("transition")
    .reset_index()
)

table = table.rename(columns={
    "MD_TO_HI": "MD→HI",
    "MD_TO_LO": "MD→LO"
})

table = table.round(4)

print(table)

## CREMA-D intensity-transition statistics

Run the following definitions and execution cells to compute centroid-proximity changes, speaker-level sign-flip tests, bootstrap intervals, and Holm-adjusted results.

Set `WITHIN_PAIRS_CSV` to reuse an exported pair-score CSV. Include `Model`, `Layer`, `Coarse`, `Transition`, `speaker`, `utterance`, `source_similarity`, `target_similarity`, and `similarity_change`, together with the CREMA-D dataset fields. The run saves pair scores, statistical summaries, matching coverage, speaker scores, centroid metadata, and a manifest under the configured output directory.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid

import numpy as np
import pandas as pd

# Optional: set this to within_cremad_pair_scores.csv exported by this section.
# Cross-dataset alignment CSVs and summary-only tables are NOT valid inputs.
WITHIN_PAIRS_CSV = globals().get("WITHIN_PAIRS_CSV", None)
WITHIN_N_SIGN_FLIPS = 19_999
WITHIN_N_BOOTSTRAP = 9_999
WITHIN_MAX_EXACT_SPEAKERS = 16
WITHIN_STATS_SEED = 20261002
WITHIN_ALPHA = 0.05

WITHIN_GROUP_COLUMNS = [
    "Reference_dataset", "Shift_dataset", "Model", "Layer", "Coarse"
]
WITHIN_TRANSITIONS = ["MD_TO_HI", "MD_TO_LO"]


def within_seed(test_key, purpose, seed):
    """Stable, separate RNG streams for every hypothesis and resampling method."""
    payload = json.dumps([int(seed), list(test_key), purpose], ensure_ascii=True)
    digest = hashlib.sha256(payload.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "little")


def validate_within_pairs(frame):
    required = WITHIN_GROUP_COLUMNS + [
        "Transition", "speaker", "utterance", "similarity_change"
    ]
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f"Within-CREMA-D pair table is missing columns: {missing}")
    frame = frame[required].copy()
    if frame.empty or frame.isna().any().any():
        raise ValueError("Within-CREMA-D pair table must be nonempty with no missing values")
    string_columns = [c for c in required if c != "similarity_change"]
    for column in string_columns:
        frame[column] = frame[column].astype(str).str.strip()
        if frame[column].eq("").any():
            raise ValueError(f"Empty value in {column}")
    frame["similarity_change"] = pd.to_numeric(
        frame["similarity_change"], errors="raise"
    ).astype(float)
    scores = frame["similarity_change"].to_numpy()
    if not np.isfinite(scores).all() or np.any(np.abs(scores) > 2 + 2e-6):
        raise ValueError("Change in cosine similarity must be finite and in [-2, 2]")
    if not set(frame["Transition"]).issubset(WITHIN_TRANSITIONS):
        raise ValueError("Only MD_TO_HI and MD_TO_LO transitions are allowed")
    pair_key = WITHIN_GROUP_COLUMNS + ["speaker", "utterance", "Transition"]
    if frame.duplicated(pair_key).any():
        raise ValueError("Duplicate model/emotion/speaker/utterance/transition rows")
    return frame.sort_values(pair_key, kind="stable").reset_index(drop=True)


def within_holm_adjust(p_values):
    """Holm family-wise correction, valid without independence between tests."""
    p_values = np.asarray(p_values, dtype=np.float64)
    if p_values.ndim != 1 or len(p_values) == 0:
        raise ValueError("Expected a nonempty one-dimensional p-value array")
    if not np.isfinite(p_values).all() or np.any((p_values < 0) | (p_values > 1)):
        raise ValueError("Invalid p-value")
    order = np.argsort(p_values, kind="stable")
    adjusted = np.maximum.accumulate(
        (len(p_values) - np.arange(len(p_values))) * p_values[order]
    )
    result = np.empty_like(p_values)
    result[order] = np.minimum(adjusted, 1.0)
    return result


def within_speaker_cluster_inference(
    values, speakers, test_key, *,
    n_sign_flips=WITHIN_N_SIGN_FLIPS,
    n_bootstrap=WITHIN_N_BOOTSTRAP,
    max_exact_speakers=WITHIN_MAX_EXACT_SPEAKERS,
    seed=WITHIN_STATS_SEED,
):
    """Two-sided speaker sign flips plus percentile speaker-bootstrap CI.

    The statistic remains the pair-weighted mean. Every row of a speaker
    receives the same random sign, and bootstrap draws retain whole speakers.
    Sign-flip validity requires independent speaker clusters whose score
    vectors are jointly sign-symmetric under H0; a zero mean alone is not enough.
    """
    values = np.asarray(values, dtype=np.float64)
    speakers = np.asarray(speakers)
    if values.ndim != 1 or speakers.ndim != 1 or len(values) != len(speakers):
        raise ValueError("Values and speakers must be equal-length vectors")
    if len(values) == 0 or not np.isfinite(values).all() or pd.isna(speakers).any():
        raise ValueError("Values and speakers must be nonempty and finite/nonmissing")
    if n_sign_flips < 1 or n_bootstrap < 2 or not 0 <= max_exact_speakers <= 20:
        raise ValueError("Invalid resampling counts or exact-enumeration limit")

    cluster_input = pd.DataFrame({"speaker": speakers.astype(str), "value": values})
    # Stable ordering also makes results invariant to input-row ordering.
    cluster_input = cluster_input.sort_values(["speaker", "value"], kind="stable")
    clusters = cluster_input.groupby("speaker", sort=True)["value"].agg(["sum", "count"])
    clusters = clusters.rename(columns={"sum": "score_sum", "count": "N_pairs"}).reset_index()
    n_speakers = len(clusters)
    if n_speakers < 2:
        raise ValueError(f"{test_key}: at least two independent speakers are required")

    sums = clusters["score_sum"].to_numpy(dtype=np.float64)
    counts = clusters["N_pairs"].to_numpy(dtype=np.int64)
    denominator = int(counts.sum())
    observed = float(sums.sum() / denominator)
    # Handle floating-point ties relative to the arithmetic scale, including
    # cancellation near zero. This cannot turn a null p-value into zero.
    tolerance = 100 * np.finfo(np.float64).eps * max(
        float(np.abs(sums).sum() / denominator), np.finfo(np.float64).tiny
    )

    exact = n_speakers <= max_exact_speakers
    n_draws = (1 << n_speakers) if exact else int(n_sign_flips)
    flip_seed = within_seed(test_key, "speaker_sign_flip", seed)
    flip_rng = np.random.default_rng(flip_seed)
    extreme = 0
    for start in range(0, n_draws, 1024):
        batch = min(1024, n_draws - start)
        if exact:
            patterns = np.arange(start, start + batch, dtype=np.uint64)[:, None]
            bits = (patterns >> np.arange(n_speakers, dtype=np.uint64)) & np.uint64(1)
            signs = 2 * bits.astype(np.int8) - 1
        else:
            signs = 2 * flip_rng.integers(0, 2, size=(batch, n_speakers), dtype=np.int8) - 1
        null_means = (signs * sums[None, :]).sum(axis=1) / denominator
        extreme += int(np.count_nonzero(np.abs(null_means) >= abs(observed) - tolerance))

    # Exact enumeration includes the observed sign pattern; do not add one.
    # Random Monte Carlo draws use the finite-resampling +1 correction.
    p_value = extreme / n_draws if exact else (extreme + 1) / (n_draws + 1)

    bootstrap_seed = within_seed(test_key, "speaker_bootstrap", seed)
    bootstrap_rng = np.random.default_rng(bootstrap_seed)
    bootstrap_means = np.empty(n_bootstrap, dtype=np.float64)
    for start in range(0, n_bootstrap, 1024):
        batch = min(1024, n_bootstrap - start)
        draw = bootstrap_rng.integers(0, n_speakers, size=(batch, n_speakers))
        bootstrap_means[start:start + batch] = (
            sums[draw].sum(axis=1) / counts[draw].sum(axis=1)
        )
    ci_low, ci_high = np.quantile(bootstrap_means, [0.025, 0.975])
    clusters["speaker_mean"] = clusters["score_sum"] / clusters["N_pairs"]

    result = {
        "estimate": observed,
        "CI95_low": float(ci_low),
        "CI95_high": float(ci_high),
        "p_raw": float(p_value),
        "N_pairs": denominator,
        "N_speakers": n_speakers,
        "min_pairs_per_speaker": int(counts.min()),
        "max_pairs_per_speaker": int(counts.max()),
        "test_method": "exact_speaker_sign_flip" if exact else "monte_carlo_speaker_sign_flip",
        "alternative": "two-sided",
        "null_value": 0.0,
        "sign_patterns_evaluated": n_draws,
        "extreme_patterns": extreme,
        "bootstrap_draws": int(n_bootstrap),
        "sign_flip_seed": flip_seed,
        "bootstrap_seed": bootstrap_seed,
        "fewer_than_10_speakers": bool(n_speakers < 10),
    }
    return result, clusters


def within_matching_audit(frame):
    """Inventory complete and incomplete matches without changing input scores."""
    frame = validate_within_pairs(frame)
    match_key = WITHIN_GROUP_COLUMNS + ["speaker", "utterance"]
    inventory = frame.pivot(
        index=match_key, columns="Transition", values="similarity_change"
    ).reindex(columns=WITHIN_TRANSITIONS)
    inventory.columns.name = None
    inventory = inventory.reset_index()
    has_hi = inventory["MD_TO_HI"].notna()
    has_lo = inventory["MD_TO_LO"].notna()
    inventory["match_status"] = np.select(
        [has_hi & has_lo, has_hi], ["complete", "HI_only"], default="LO_only"
    )
    # Missing counterparts stay missing. Never fill with zero or match row order.
    inventory["HI_minus_LO"] = inventory["MD_TO_HI"] - inventory["MD_TO_LO"]
    coverage_rows = []
    for keys, group in inventory.groupby(WITHIN_GROUP_COLUMNS, sort=True):
        complete = group["match_status"].eq("complete")
        coverage_rows.append({
            **dict(zip(WITHIN_GROUP_COLUMNS, keys)),
            "N_HI_available": int(group["MD_TO_HI"].notna().sum()),
            "N_LO_available": int(group["MD_TO_LO"].notna().sum()),
            "N_complete_pairs": int(complete.sum()),
            "N_matched_speakers": int(group.loc[complete, "speaker"].nunique()),
            "N_HI_only": int(group["match_status"].eq("HI_only").sum()),
            "N_LO_only": int(group["match_status"].eq("LO_only").sum()),
        })
    return inventory, pd.DataFrame(coverage_rows)


def within_cremad_inference(frame, *, alpha=WITHIN_ALPHA, **resampling):
    """Test all available cosine changes; paired contrasts use complete matches.

    Incomplete matching never blocks the cosine-change-versus-zero tests. Contrasts
    with fewer than two matched speakers are explicitly marked not tested.
    Three hypotheses per observed model/layer/emotion group form ONE planned
    Holm family, including unavailable hypotheses conservatively as p=1 only
    for adjustment. Unavailable p-values remain missing in the output.
    """
    if not 0 < alpha < 1:
        raise ValueError("alpha must be between zero and one")
    frame = validate_within_pairs(frame)
    match_inventory, coverage = within_matching_audit(frame)
    matched = match_inventory.loc[match_inventory["match_status"].eq("complete")].copy()
    matched = matched.drop(columns="match_status").reset_index(drop=True)
    matched_groups = {
        keys: group for keys, group in matched.groupby(WITHIN_GROUP_COLUMNS, sort=True)
    }

    results, speaker_tables = [], []

    def run_group(group, keys, test, transition, score_column):
        meta = dict(zip(WITHIN_GROUP_COLUMNS, keys))
        meta.update({
            "Test": test, "Transition": transition,
            "analysis_population": (
                "all_available_transition_pairs" if test == "centroid_change_vs_zero"
                else "complete_HI_LO_matches_only"
            ),
        })
        n_speakers = int(group["speaker"].nunique())
        if n_speakers < 2:
            reason = (
                "not_tested_no_matched_pairs" if group.empty and test == "paired_HI_minus_LO"
                else "not_tested_no_rows" if group.empty
                else "not_tested_fewer_than_2_speakers"
            )
            results.append({
                **meta, "status": reason,
                "estimate": float(group[score_column].mean()) if len(group) else np.nan,
                "CI95_low": np.nan, "CI95_high": np.nan, "p_raw": np.nan,
                "N_pairs": len(group), "N_speakers": n_speakers,
                "test_method": "not_tested", "alternative": "two-sided", "null_value": 0.0,
                "sign_patterns_evaluated": 0, "bootstrap_draws": 0,
                "fewer_than_10_speakers": bool(n_speakers < 10),
            })
            return
        result, speakers = within_speaker_cluster_inference(
            group[score_column].to_numpy(),
            group["speaker"].to_numpy(),
            tuple(keys) + (test, transition),
            **resampling,
        )
        results.append({**meta, "status": "tested", **result})
        for name, value in meta.items():
            speakers[name] = value
        speaker_tables.append(speakers)

    for keys, group in frame.groupby(WITHIN_GROUP_COLUMNS, sort=True):
        for transition in WITHIN_TRANSITIONS:
            available = group.loc[group["Transition"].eq(transition)]
            run_group(available, keys, "centroid_change_vs_zero", transition, "similarity_change")
        complete = matched_groups.get(keys, matched.iloc[:0])
        run_group(complete, keys, "paired_HI_minus_LO", "MD_TO_HI_minus_MD_TO_LO", "HI_minus_LO")

    result_rows = results
    results = pd.DataFrame(result_rows)
    # Nullable uint64 retains large seeds exactly even when skipped rows have no seed.
    for seed_column in ["sign_flip_seed", "bootstrap_seed"]:
        results[seed_column] = pd.array(
            [row.get(seed_column, pd.NA) for row in result_rows], dtype="UInt64"
        )
    tested = results["status"].eq("tested")
    results["p_holm"] = within_holm_adjust(results["p_raw"].fillna(1.0).to_numpy())
    results.loc[~tested, "p_holm"] = np.nan
    results["holm_family_size"] = len(results)
    results["N_tests_performed"] = int(tested.sum())
    results["reject_holm_0_05"] = pd.array(results["p_holm"] <= alpha, dtype="boolean")
    results.loc[~tested, "reject_holm_0_05"] = pd.NA
    # The column is explicitly named by the configured threshold below.
    if alpha != 0.05:
        results = results.rename(columns={"reject_holm_0_05": "reject_holm"})
    results["alpha"] = float(alpha)
    results = results.merge(coverage, on=WITHIN_GROUP_COLUMNS, how="left", validate="many_to_one")
    results = results.sort_values(
        ["Test"] + WITHIN_GROUP_COLUMNS + ["Transition"], kind="stable"
    ).reset_index(drop=True)
    speakers = (
        pd.concat(speaker_tables, ignore_index=True) if speaker_tables
        else pd.DataFrame(columns=["speaker", "score_sum", "N_pairs", "speaker_mean"]
                          + WITHIN_GROUP_COLUMNS + ["Test", "Transition", "analysis_population"])
    )
    return results, matched, speakers


# Paths copied from geometry.ipynb, cells 2 and 12; no path remapping.
WITHIN_REFERENCE_CSVS = {
    'QWEN': '/content/drive/MyDrive/probe/probe_representations_2/probe_rep_index.csv',
    'FLAMINGO': '/content/drive/MyDrive/probe/probe_representations_flamingo_2/probe_rep_index.csv',
}
WITHIN_TRANSITION_CSVS = {
    'QWEN': '/content/drive/MyDrive/probing/full_probe/albe/merged_results.csv',
    'FLAMINGO': '/content/drive/MyDrive/probing/full_probe_flamingo/abla/merged_results.csv',
}
WITHIN_MODEL_NAMES = {'QWEN': 'Qwen2-Audio', 'FLAMINGO': 'Audio-Flamingo-3'}
WITHIN_COARSE = ['ANG', 'DIS', 'FEA', 'HAP', 'SAD']
WITHIN_LAYER = 'lm_2'


def within_normalized_vector(value):
    """Same float32 pooling and normalization as the original process_rep."""
    if hasattr(value, 'detach'):
        value = value.detach().cpu().float().numpy()
    x = np.asarray(value, dtype=np.float32)
    if x.ndim == 3 and x.shape[0] == 1:
        x = x.mean(axis=1).squeeze(0)
    elif x.ndim == 2 and x.shape[0] == 1:
        x = x.squeeze(0)
    if x.ndim != 1 or x.size == 0 or not np.isfinite(x).all():
        raise ValueError(f'Invalid representation shape/values: {x.shape}')
    norm = np.linalg.norm(x)
    if not np.isfinite(norm) or norm <= 0:
        raise ValueError('Zero or invalid representation norm')
    return x / (norm + 1e-8)


def within_load_embedding(path, layer):
    path = Path(path)
    if path.suffix == '.npz':
        with np.load(path, allow_pickle=False) as data:
            if layer not in data:
                raise KeyError(f'{path}: missing layer {layer}')
            return within_normalized_vector(data[layer])
    import torch
    # Refuse executable pickle payloads. Standard tensor dictionaries work.
    data = torch.load(path, map_location='cpu', weights_only=True)
    if not isinstance(data, dict) or layer not in data:
        raise KeyError(f'{path}: missing layer {layer}')
    return within_normalized_vector(data[layer])


def within_parse_audio(value):
    name = Path(str(value).strip().replace('\\', '/')).stem
    parts = name.split('_')
    if len(parts) != 4 or any(not x for x in parts):
        raise ValueError(f'Invalid CREMA-D audio filename: {value!r}')
    return dict(zip(['speaker', 'utterance', 'coarse', 'intensity'], parts))


def within_load_transition_manifest():
    rows, inputs = [], []
    for model, filename in WITHIN_TRANSITION_CSVS.items():
        path = Path(filename)
        frame = pd.read_csv(path, dtype={'id': str})
        required = {'task', 'neutral_path', 'manipulated_path', 'neutral_rep_path', 'rep_path'}
        if not required.issubset(frame.columns):
            raise ValueError(f'{path}: missing {sorted(required - set(frame.columns))}')
        frame = frame.loc[frame['task'].astype(str).str.strip().str.lower().eq('intensity')]
        for _, row in frame.iterrows():
            source = within_parse_audio(row['neutral_path'])
            target = within_parse_audio(row['manipulated_path'])
            if any(source[k] != target[k] for k in ['speaker', 'utterance', 'coarse']):
                raise ValueError(f'Unmatched source/target in {path}: {row.to_dict()}')
            transition = source['intensity'] + '_TO_' + target['intensity']
            if transition not in WITHIN_TRANSITIONS or source['coarse'] not in WITHIN_COARSE:
                continue
            rows.append({'model': model, 'speaker': source['speaker'],
                         'utterance': source['utterance'], 'coarse': source['coarse'],
                         'transition': transition, 'source_rep': row['neutral_rep_path'],
                         'target_rep': row['rep_path']})
        inputs.append({'path': str(path), 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
    return pd.DataFrame(rows), inputs


def within_load_reference_centroids(loader=within_load_embedding):
    centroids, inventory, inputs = {}, [], []
    for model, filename in WITHIN_REFERENCE_CSVS.items():
        path = Path(filename)
        frame = pd.read_csv(path)
        if not {'emotion', 'rep_path'}.issubset(frame.columns):
            raise ValueError(f'{path}: requires emotion and rep_path')
        if frame[['emotion', 'rep_path']].isna().any().any():
            raise ValueError(f'{path}: missing emotion or representation path')
        frame['emotion'] = frame['emotion'].astype(str).str.strip().str.upper()
        frame['rep_path'] = frame['rep_path'].astype(str).str.strip()
        if frame['rep_path'].duplicated().any():
            raise ValueError(f'{path}: duplicate reference representation paths')
        centroids[model] = {}
        for emotion, group in frame.groupby('emotion', sort=True):
            vectors = [loader(row['rep_path'], WITHIN_LAYER) for _, row in group.iterrows()]
            mean = np.stack(vectors).mean(axis=0)
            centroids[model][emotion] = within_normalized_vector(mean)
            inventory.append({'Model': WITHIN_MODEL_NAMES[model], 'Layer': WITHIN_LAYER,
                              'Coarse': emotion, 'N_reference': len(group), 'reference_csv': str(path)})
            print(f'Reference centroids: {model}/{emotion}: {len(group)} vectors')
        inputs.append({'path': str(path), 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
    return centroids, pd.DataFrame(inventory), inputs


def within_scores_from_transitions(transitions, centroids, loader=within_load_embedding):
    required = ['model', 'speaker', 'utterance', 'coarse', 'transition', 'source_rep', 'target_rep']
    if not set(required).issubset(transitions.columns):
        raise ValueError(f'transition_df lacks {sorted(set(required) - set(transitions.columns))}')
    frame = transitions[required].copy()
    if frame.isna().any().any():
        raise ValueError('Missing transition metadata; no rows were dropped')
    for c in required:
        frame[c] = frame[c].astype(str).str.strip()
        if frame[c].eq('').any():
            raise ValueError(f'Empty transition field: {c}')
    frame = frame.loc[frame['transition'].isin(WITHIN_TRANSITIONS) & frame['coarse'].isin(WITHIN_COARSE)]
    keys = ['model', 'coarse', 'speaker', 'utterance', 'transition']
    if frame.empty or frame.duplicated(keys).any():
        raise ValueError('Empty transition table or duplicate pair identities')
    if set(frame['model']) != set(WITHIN_MODEL_NAMES):
        raise ValueError('Expected the original QWEN and FLAMINGO model keys')
    # The same MD anchor must be used for both target intensities.
    anchors = frame.groupby(['model', 'coarse', 'speaker', 'utterance'])['source_rep'].nunique()
    if anchors.gt(1).any():
        raise ValueError('HI/LO rows with one match key use different MD source files')
    expected = pd.MultiIndex.from_product([list(WITHIN_MODEL_NAMES), WITHIN_COARSE, WITHIN_TRANSITIONS])
    counts = frame.groupby(['model', 'coarse', 'transition']).size().reindex(expected, fill_value=0)
    if not counts.eq(42).all():
        raise ValueError(f'The original design has 42 pairs per cell; found {counts.to_dict()}')
    rows = []
    for model, group in frame.groupby('model', sort=True):
        cache = {}
        def embedding(path):
            if path not in cache:
                value = loader(path, WITHIN_LAYER)
                if value is None:
                    raise ValueError(f'Missing {WITHIN_LAYER} in {path}; no rows were dropped')
                cache[path] = within_normalized_vector(value)
            return cache[path]
        for index, (_, row) in enumerate(group.iterrows(), 1):
            centroid = within_normalized_vector(centroids[model][row['coarse']])
            source, target = embedding(row['source_rep']), embedding(row['target_rep'])
            if source.shape != centroid.shape or target.shape != centroid.shape:
                raise ValueError(f'Embedding/centroid dimension mismatch for {model}/{row["coarse"]}')
            # Explicit cosine, matching sklearn cosine_similarity in original cell 13.
            source_similarity = float(np.dot(source / np.linalg.norm(source), centroid / np.linalg.norm(centroid)))
            target_similarity = float(np.dot(target / np.linalg.norm(target), centroid / np.linalg.norm(centroid)))
            rows.append({'Reference_dataset': 'CREMA-D', 'Shift_dataset': 'CREMA-D',
                         'Model': WITHIN_MODEL_NAMES[model], 'Layer': WITHIN_LAYER,
                         'Coarse': row['coarse'], 'Transition': row['transition'],
                         'speaker': row['speaker'], 'utterance': row['utterance'],
                         'source_rep': row['source_rep'], 'target_rep': row['target_rep'],
                         'source_similarity': source_similarity, 'target_similarity': target_similarity,
                         'similarity_change': target_similarity - source_similarity})
            if index % 100 == 0 or index == len(group):
                print(f'Within-CREMA-D scores: {model}: {index}/{len(group)}')
    return pd.DataFrame(rows)


def within_prepare_scores(frame):
    """Validate provenance, paired IDs, cosine bounds and the subtraction identity."""
    frame = frame.copy()
    for column in ['Reference_dataset', 'Shift_dataset']:
        if column not in frame or not frame[column].eq('CREMA-D').all():
            raise ValueError(f'{column} must explicitly be CREMA-D; cross-dataset CSVs are invalid')
    required = ['source_similarity', 'target_similarity', 'similarity_change']
    if not set(required).issubset(frame.columns):
        raise ValueError('Need per-pair source_similarity, target_similarity and similarity_change; summary means are insufficient')
    for column in required:
        frame[column] = pd.to_numeric(frame[column], errors='raise').astype(float)
        if not np.isfinite(frame[column]).all():
            raise ValueError(f'Invalid {column}')
    if (frame[['source_similarity', 'target_similarity']].abs() > 1 + 1e-6).any().any():
        raise ValueError('Source/target cosine must be in [-1, 1]')
    if not np.allclose(frame['similarity_change'], frame['target_similarity'] - frame['source_similarity'], atol=1e-7, rtol=1e-6):
        raise ValueError('similarity_change must equal target_similarity minus source_similarity')
    # This checks identity completeness and score bounds; retain richer provenance.
    validate_within_pairs(frame)
    for column in WITHIN_GROUP_COLUMNS + ['Transition', 'speaker', 'utterance']:
        frame[column] = frame[column].astype(str).str.strip()
    expected = {(m, WITHIN_LAYER, e) for m in WITHIN_MODEL_NAMES.values() for e in WITHIN_COARSE}
    actual = set(frame[['Model', 'Layer', 'Coarse']].itertuples(index=False, name=None))
    if actual != expected:
        raise ValueError(f'Expected all 10 model/layer/emotion groups for the planned 30 tests. Missing={expected-actual}; unexpected={actual-expected}')
    key = WITHIN_GROUP_COLUMNS + ['speaker', 'utterance']
    spread = frame.groupby(key)['source_similarity'].agg(['min', 'max'])
    if (spread['max'] - spread['min'] > 1e-6).any():
        raise ValueError('Matched HI/LO rows have inconsistent MD source similarity')
    if 'source_rep' in frame and frame.groupby(key)['source_rep'].nunique().gt(1).any():
        raise ValueError('Matched HI/LO rows have inconsistent MD source paths')
    return frame.sort_values(key + ['Transition'], kind='stable').reset_index(drop=True)


In [ ]:
_within_inventory = pd.DataFrame()
_within_inputs = []
_within_centroids = None
if WITHIN_PAIRS_CSV is not None:
    _within_csv = Path(WITHIN_PAIRS_CSV).expanduser()
    _within_input = pd.read_csv(_within_csv, dtype={c: str for c in WITHIN_GROUP_COLUMNS + ['speaker', 'utterance', 'Transition']})
    _within_source = str(_within_csv.resolve())
    _within_inputs.append({'path': _within_source, 'sha256': hashlib.sha256(_within_csv.read_bytes()).hexdigest()})
elif 'within_cremad_pair_scores' in globals():
    _within_input = within_cremad_pair_scores.copy()
    _within_source = 'in-memory within_cremad_pair_scores with explicit pair IDs'
else:
    if 'transition_df' in globals():
        _within_transitions = transition_df.copy()
        _within_inputs.append({'source': 'in-memory transition_df from original geometry.ipynb'})
    else:
        _missing = [p for p in WITHIN_TRANSITION_CSVS.values() if not Path(p).is_file()]
        if _missing:
            raise FileNotFoundError('The raw data are on Google Drive. In Colab, mount the original Drive, then run this section; or set WITHIN_PAIRS_CSV to within_cremad_pair_scores.csv. Missing: ' + repr(_missing))
        _within_transitions, _within_inputs = within_load_transition_manifest()
    if 'model_centroids' in globals():
        if globals().get('TARGET_LAYER') != WITHIN_LAYER:
            raise ValueError('Cannot verify that the in-memory model_centroids use lm_2')
        _within_centroids = model_centroids
        _within_inputs.append({'source': 'in-memory model_centroids from original cell 12', 'reference_uncertainty': 'fixed'})
    else:
        _within_centroids, _within_inventory, _within_reference_inputs = within_load_reference_centroids()
        _within_inputs.extend(_within_reference_inputs)
    _within_input = within_scores_from_transitions(_within_transitions, _within_centroids)
    _within_source = 'recomputed per-pair similarities using the original within-CREMA-D metric'

within_cremad_pair_scores = within_prepare_scores(_within_input)
within_cremad_statistics, within_cremad_matched_pairs, within_cremad_speaker_scores = within_cremad_inference(within_cremad_pair_scores)
within_cremad_match_inventory, within_cremad_matching_coverage = within_matching_audit(within_cremad_pair_scores)
assert len(within_cremad_statistics) == 30

if 'RUN_DIR' in globals():
    _within_parent = Path(RUN_DIR)
elif WITHIN_PAIRS_CSV is not None:
    _within_parent = Path(WITHIN_PAIRS_CSV).expanduser().resolve().parent
elif Path('/content/drive/MyDrive').is_dir():
    _within_parent = Path('/content/drive/MyDrive/subprobe/geometry_unified')
else:
    _within_parent = Path.cwd() / 'geometry_unified'
_within_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
WITHIN_STATS_DIR = _within_parent / f'within_cremad_statistics_{_within_stamp}_{uuid.uuid4().hex[:8]}'
WITHIN_STATS_DIR.mkdir(parents=True, exist_ok=False)
for filename, table in {
    'within_cremad_pair_scores.csv': within_cremad_pair_scores,
    'within_cremad_statistics.csv': within_cremad_statistics,
    'within_cremad_matched_pairs.csv': within_cremad_matched_pairs,
    'within_cremad_speaker_scores.csv': within_cremad_speaker_scores,
    'within_cremad_match_inventory.csv': within_cremad_match_inventory,
    'within_cremad_matching_coverage.csv': within_cremad_matching_coverage,
}.items():
    table.to_csv(WITHIN_STATS_DIR / filename, index=False)
if not _within_inventory.empty:
    _within_inventory.to_csv(WITHIN_STATS_DIR / 'within_cremad_reference_inventory.csv', index=False)
if _within_centroids is not None:
    np.savez_compressed(WITHIN_STATS_DIR / 'within_cremad_reference_centroids.npz',
                        **{model + '__' + emotion: vector for model, group in _within_centroids.items() for emotion, vector in group.items()})

within_cremad_manifest = {
    'created_utc': _within_stamp, 'input_source': _within_source, 'input_metadata': _within_inputs,
    'exported_pair_sha256': hashlib.sha256((WITHIN_STATS_DIR / 'within_cremad_pair_scores.csv').read_bytes()).hexdigest(),
    'metric': 'cos(target, fixed coarse centroid) - cos(MD, fixed coarse centroid)',
    'reference_construction': 'expected original geometry.ipynb construction: float32 normalized reference vectors, arithmetic mean by coarse emotion, normalize centroid; not independently verified for precomputed CSV inputs',
    'conditioning': 'fixed CREMA-D reference centroids; no resampling of reference estimation uncertainty; reference/evaluation speaker independence not verified',
    'statistic': 'pair-weighted mean change, or mean HI-minus-LO change in complete matched subsets',
    'resampling_unit': 'speaker; all observations from a speaker move together',
    'nulls': ['mean cosine change = 0 per transition', 'mean matched HI-minus-LO change = 0'],
    'alternative': 'two-sided, absolute mean',
    'assumptions': 'independent speaker clusters and sign-symmetric cluster score vectors under H0; zero mean alone is insufficient',
    'n_sign_flips': WITHIN_N_SIGN_FLIPS, 'exact_if_speakers_at_most': WITHIN_MAX_EXACT_SPEAKERS,
    'p_value': 'exact extreme/all patterns; Monte Carlo (extreme+1)/(draws+1)',
    'n_bootstrap': WITHIN_N_BOOTSTRAP, 'confidence_intervals': 'pointwise 95% percentile whole-speaker bootstrap',
    'seed': WITHIN_STATS_SEED, 'alpha': WITHIN_ALPHA, 'holm_family_size': 30,
    'multiple_testing_scope': 'this within-CREMA-D family only, not jointly with the previous cross-dataset tests',
    'matching': 'same model, layer, emotion, speaker, utterance and MD anchor; unmatched rows retained for zero tests only',
    'unavailable_tests': 'not tested with fewer than 2 speakers; p=1 only internally for Holm; exported p-values stay missing',
    'n_tests_performed': int(within_cremad_statistics['status'].eq('tested').sum()),
    'python': __import__('platform').python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
}
(WITHIN_STATS_DIR / 'within_cremad_statistics_manifest.json').write_text(json.dumps(within_cremad_manifest, indent=2), encoding='utf-8')
try:
    from IPython.display import display as _within_display
except ImportError:
    _within_display = print
_within_columns = ['Model', 'Layer', 'Coarse', 'Transition', 'status', 'estimate', 'CI95_low', 'CI95_high',
                   'N_pairs', 'N_speakers', 'p_raw', 'p_holm', 'reject_holm_0_05']
for _within_test in ['centroid_change_vs_zero', 'paired_HI_minus_LO']:
    print(_within_test)
    _within_display(within_cremad_statistics.loc[within_cremad_statistics['Test'].eq(_within_test), _within_columns].round(6))
print('Matching coverage:')
_within_display(within_cremad_matching_coverage)
print('30 planned within-CREMA-D tests; pointwise bootstrap CIs; Holm alpha=0.05.')
print('Unavailable tests are NOT nonsignificant findings. Inspect status and N_speakers.')
print('Results CSV:', WITHIN_STATS_DIR / 'within_cremad_statistics.csv')
print('Reusable pair CSV:', WITHIN_STATS_DIR / 'within_cremad_pair_scores.csv')


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid

import numpy as np
import pandas as pd

# Optional: set this to within_cremad_pair_scores.csv exported by this section.
# Cross-dataset alignment CSVs and summary-only tables are NOT valid inputs.
WITHIN_PAIRS_CSV = globals().get("WITHIN_PAIRS_CSV", None)
WITHIN_N_SIGN_FLIPS = 19_999
WITHIN_N_BOOTSTRAP = 9_999
WITHIN_MAX_EXACT_SPEAKERS = 16
WITHIN_STATS_SEED = 20261002
WITHIN_ALPHA = 0.05

WITHIN_GROUP_COLUMNS = [
    "Reference_dataset", "Shift_dataset", "Model", "Layer", "Coarse"
]
WITHIN_TRANSITIONS = ["MD_TO_HI", "MD_TO_LO"]


def within_seed(test_key, purpose, seed):
    """Stable, separate RNG streams for every hypothesis and resampling method."""
    payload = json.dumps([int(seed), list(test_key), purpose], ensure_ascii=True)
    digest = hashlib.sha256(payload.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "little")


def validate_within_pairs(frame):
    required = WITHIN_GROUP_COLUMNS + [
        "Transition", "speaker", "utterance", "similarity_change"
    ]
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f"Within-CREMA-D pair table is missing columns: {missing}")
    frame = frame[required].copy()
    if frame.empty or frame.isna().any().any():
        raise ValueError("Within-CREMA-D pair table must be nonempty with no missing values")
    string_columns = [c for c in required if c != "similarity_change"]
    for column in string_columns:
        frame[column] = frame[column].astype(str).str.strip()
        if frame[column].eq("").any():
            raise ValueError(f"Empty value in {column}")
    frame["similarity_change"] = pd.to_numeric(
        frame["similarity_change"], errors="raise"
    ).astype(float)
    scores = frame["similarity_change"].to_numpy()
    if not np.isfinite(scores).all() or np.any(np.abs(scores) > 2 + 2e-6):
        raise ValueError("Change in cosine similarity must be finite and in [-2, 2]")
    if not set(frame["Transition"]).issubset(WITHIN_TRANSITIONS):
        raise ValueError("Only MD_TO_HI and MD_TO_LO transitions are allowed")
    pair_key = WITHIN_GROUP_COLUMNS + ["speaker", "utterance", "Transition"]
    if frame.duplicated(pair_key).any():
        raise ValueError("Duplicate model/emotion/speaker/utterance/transition rows")
    return frame.sort_values(pair_key, kind="stable").reset_index(drop=True)


def within_holm_adjust(p_values):
    """Holm family-wise correction, valid without independence between tests."""
    p_values = np.asarray(p_values, dtype=np.float64)
    if p_values.ndim != 1 or len(p_values) == 0:
        raise ValueError("Expected a nonempty one-dimensional p-value array")
    if not np.isfinite(p_values).all() or np.any((p_values < 0) | (p_values > 1)):
        raise ValueError("Invalid p-value")
    order = np.argsort(p_values, kind="stable")
    adjusted = np.maximum.accumulate(
        (len(p_values) - np.arange(len(p_values))) * p_values[order]
    )
    result = np.empty_like(p_values)
    result[order] = np.minimum(adjusted, 1.0)
    return result


def within_speaker_cluster_inference(
    values, speakers, test_key, *,
    n_sign_flips=WITHIN_N_SIGN_FLIPS,
    n_bootstrap=WITHIN_N_BOOTSTRAP,
    max_exact_speakers=WITHIN_MAX_EXACT_SPEAKERS,
    seed=WITHIN_STATS_SEED,
):
    """Two-sided speaker sign flips plus percentile speaker-bootstrap CI.

    The statistic remains the pair-weighted mean. Every row of a speaker
    receives the same random sign, and bootstrap draws retain whole speakers.
    Sign-flip validity requires independent speaker clusters whose score
    vectors are jointly sign-symmetric under H0; a zero mean alone is not enough.
    """
    values = np.asarray(values, dtype=np.float64)
    speakers = np.asarray(speakers)
    if values.ndim != 1 or speakers.ndim != 1 or len(values) != len(speakers):
        raise ValueError("Values and speakers must be equal-length vectors")
    if len(values) == 0 or not np.isfinite(values).all() or pd.isna(speakers).any():
        raise ValueError("Values and speakers must be nonempty and finite/nonmissing")
    if n_sign_flips < 1 or n_bootstrap < 2 or not 0 <= max_exact_speakers <= 20:
        raise ValueError("Invalid resampling counts or exact-enumeration limit")

    cluster_input = pd.DataFrame({"speaker": speakers.astype(str), "value": values})
    # Stable ordering also makes results invariant to input-row ordering.
    cluster_input = cluster_input.sort_values(["speaker", "value"], kind="stable")
    clusters = cluster_input.groupby("speaker", sort=True)["value"].agg(["sum", "count"])
    clusters = clusters.rename(columns={"sum": "score_sum", "count": "N_pairs"}).reset_index()
    n_speakers = len(clusters)
    if n_speakers < 2:
        raise ValueError(f"{test_key}: at least two independent speakers are required")

    sums = clusters["score_sum"].to_numpy(dtype=np.float64)
    counts = clusters["N_pairs"].to_numpy(dtype=np.int64)
    denominator = int(counts.sum())
    observed = float(sums.sum() / denominator)
    # Handle floating-point ties relative to the arithmetic scale, including
    # cancellation near zero. This cannot turn a null p-value into zero.
    tolerance = 100 * np.finfo(np.float64).eps * max(
        float(np.abs(sums).sum() / denominator), np.finfo(np.float64).tiny
    )

    exact = n_speakers <= max_exact_speakers
    n_draws = (1 << n_speakers) if exact else int(n_sign_flips)
    flip_seed = within_seed(test_key, "speaker_sign_flip", seed)
    flip_rng = np.random.default_rng(flip_seed)
    extreme = 0
    for start in range(0, n_draws, 1024):
        batch = min(1024, n_draws - start)
        if exact:
            patterns = np.arange(start, start + batch, dtype=np.uint64)[:, None]
            bits = (patterns >> np.arange(n_speakers, dtype=np.uint64)) & np.uint64(1)
            signs = 2 * bits.astype(np.int8) - 1
        else:
            signs = 2 * flip_rng.integers(0, 2, size=(batch, n_speakers), dtype=np.int8) - 1
        null_means = (signs * sums[None, :]).sum(axis=1) / denominator
        extreme += int(np.count_nonzero(np.abs(null_means) >= abs(observed) - tolerance))

    # Exact enumeration includes the observed sign pattern; do not add one.
    # Random Monte Carlo draws use the finite-resampling +1 correction.
    p_value = extreme / n_draws if exact else (extreme + 1) / (n_draws + 1)

    bootstrap_seed = within_seed(test_key, "speaker_bootstrap", seed)
    bootstrap_rng = np.random.default_rng(bootstrap_seed)
    bootstrap_means = np.empty(n_bootstrap, dtype=np.float64)
    for start in range(0, n_bootstrap, 1024):
        batch = min(1024, n_bootstrap - start)
        draw = bootstrap_rng.integers(0, n_speakers, size=(batch, n_speakers))
        bootstrap_means[start:start + batch] = (
            sums[draw].sum(axis=1) / counts[draw].sum(axis=1)
        )
    ci_low, ci_high = np.quantile(bootstrap_means, [0.025, 0.975])
    clusters["speaker_mean"] = clusters["score_sum"] / clusters["N_pairs"]

    result = {
        "estimate": observed,
        "CI95_low": float(ci_low),
        "CI95_high": float(ci_high),
        "p_raw": float(p_value),
        "N_pairs": denominator,
        "N_speakers": n_speakers,
        "min_pairs_per_speaker": int(counts.min()),
        "max_pairs_per_speaker": int(counts.max()),
        "test_method": "exact_speaker_sign_flip" if exact else "monte_carlo_speaker_sign_flip",
        "alternative": "two-sided",
        "null_value": 0.0,
        "sign_patterns_evaluated": n_draws,
        "extreme_patterns": extreme,
        "bootstrap_draws": int(n_bootstrap),
        "sign_flip_seed": flip_seed,
        "bootstrap_seed": bootstrap_seed,
        "fewer_than_10_speakers": bool(n_speakers < 10),
    }
    return result, clusters



# Paths copied from geometry.ipynb, cells 2 and 12; no path remapping.
WITHIN_REFERENCE_CSVS = {
    'QWEN': '/content/drive/MyDrive/probe/probe_representations_2/probe_rep_index.csv',
    'FLAMINGO': '/content/drive/MyDrive/probe/probe_representations_flamingo_2/probe_rep_index.csv',
}
WITHIN_TRANSITION_CSVS = {
    'QWEN': '/content/drive/MyDrive/probing/full_probe/albe/merged_results.csv',
    'FLAMINGO': '/content/drive/MyDrive/probing/full_probe_flamingo/abla/merged_results.csv',
}
WITHIN_MODEL_NAMES = {'QWEN': 'Qwen2-Audio', 'FLAMINGO': 'Audio-Flamingo-3'}
WITHIN_COARSE_TAXONOMY = ['ANG', 'DIS', 'FEA', 'HAP', 'SAD', 'SUR', 'NEU']
WITHIN_REFERENCE_EMOTIONS = ['ANG', 'DIS', 'FEA', 'HAP', 'SAD', 'NEU']
WITHIN_COARSE = ['ANG', 'DIS', 'FEA', 'HAP', 'SAD']
WITHIN_LAYER = 'lm_2'


def within_normalized_vector(value):
    """Same float32 pooling and normalization as the original process_rep."""
    if hasattr(value, 'detach'):
        value = value.detach().cpu().float().numpy()
    x = np.asarray(value, dtype=np.float32)
    if x.ndim == 3 and x.shape[0] == 1:
        x = x.mean(axis=1).squeeze(0)
    elif x.ndim == 2 and x.shape[0] == 1:
        x = x.squeeze(0)
    if x.ndim != 1 or x.size == 0 or not np.isfinite(x).all():
        raise ValueError(f'Invalid representation shape/values: {x.shape}')
    norm = np.linalg.norm(x)
    if not np.isfinite(norm) or norm <= 0:
        raise ValueError('Zero or invalid representation norm')
    return x / (norm + 1e-8)


def within_load_embedding(path, layer):
    path = Path(path)
    if path.suffix == '.npz':
        with np.load(path, allow_pickle=False) as data:
            if layer not in data:
                raise KeyError(f'{path}: missing layer {layer}')
            return within_normalized_vector(data[layer])
    import torch
    # Refuse executable pickle payloads. Standard tensor dictionaries work.
    data = torch.load(path, map_location='cpu', weights_only=True)
    if not isinstance(data, dict) or layer not in data:
        raise KeyError(f'{path}: missing layer {layer}')
    return within_normalized_vector(data[layer])


def within_parse_audio(value):
    name = Path(str(value).strip().replace('\\', '/')).stem
    parts = name.split('_')
    if len(parts) != 4 or any(not x for x in parts):
        raise ValueError(f'Invalid CREMA-D audio filename: {value!r}')
    return dict(zip(['speaker', 'utterance', 'coarse', 'intensity'], parts))


def within_load_transition_manifest():
    rows, inputs = [], []
    for model, filename in WITHIN_TRANSITION_CSVS.items():
        path = Path(filename)
        frame = pd.read_csv(path, dtype={'id': str})
        required = {'task', 'neutral_path', 'manipulated_path', 'neutral_rep_path', 'rep_path'}
        if not required.issubset(frame.columns):
            raise ValueError(f'{path}: missing {sorted(required - set(frame.columns))}')
        frame = frame.loc[frame['task'].astype(str).str.strip().str.lower().eq('intensity')]
        for _, row in frame.iterrows():
            source = within_parse_audio(row['neutral_path'])
            target = within_parse_audio(row['manipulated_path'])
            if any(source[k] != target[k] for k in ['speaker', 'utterance', 'coarse']):
                raise ValueError(f'Unmatched source/target in {path}: {row.to_dict()}')
            if source['coarse'] not in WITHIN_REFERENCE_EMOTIONS:
                raise ValueError(f'Target emotion is outside the CREMA-D reference inventory: {source["coarse"]}')
            transition = source['intensity'] + '_TO_' + target['intensity']
            if transition not in WITHIN_TRANSITIONS or source['coarse'] not in WITHIN_COARSE:
                continue
            rows.append({'model': model, 'speaker': source['speaker'],
                         'utterance': source['utterance'], 'coarse': source['coarse'],
                         'transition': transition, 'source_rep': row['neutral_rep_path'],
                         'target_rep': row['rep_path']})
        inputs.append({'path': str(path), 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
    return pd.DataFrame(rows), inputs


def within_load_reference_centroids(loader=within_load_embedding):
    centroids, inventory, inputs = {}, [], []
    for model, filename in WITHIN_REFERENCE_CSVS.items():
        path = Path(filename)
        frame = pd.read_csv(path)
        if not {'emotion', 'rep_path'}.issubset(frame.columns):
            raise ValueError(f'{path}: requires emotion and rep_path')
        if frame[['emotion', 'rep_path']].isna().any().any():
            raise ValueError(f'{path}: missing emotion or representation path')
        frame['emotion'] = frame['emotion'].astype(str).str.strip().str.upper()
        if not set(frame['emotion']).issubset(WITHIN_REFERENCE_EMOTIONS):
            raise ValueError(f'{path}: labels outside the six CREMA-D reference emotions')
        frame['rep_path'] = frame['rep_path'].astype(str).str.strip()
        if frame['rep_path'].duplicated().any():
            raise ValueError(f'{path}: duplicate reference representation paths')
        centroids[model] = {}
        for emotion, group in frame.groupby('emotion', sort=True):
            vectors = [loader(row['rep_path'], WITHIN_LAYER) for _, row in group.iterrows()]
            mean = np.stack(vectors).mean(axis=0)
            centroids[model][emotion] = within_normalized_vector(mean)
            inventory.append({'Model': WITHIN_MODEL_NAMES[model], 'Layer': WITHIN_LAYER,
                              'Coarse': emotion, 'N_reference': len(group), 'reference_csv': str(path)})
            print(f'Reference centroids: {model}/{emotion}: {len(group)} vectors')
        inputs.append({'path': str(path), 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
    return centroids, pd.DataFrame(inventory), inputs


def within_scores_from_transitions(transitions, centroids, loader=within_load_embedding):
    required = ['model', 'speaker', 'utterance', 'coarse', 'transition', 'source_rep', 'target_rep']
    if not set(required).issubset(transitions.columns):
        raise ValueError(f'transition_df lacks {sorted(set(required) - set(transitions.columns))}')
    frame = transitions[required].copy()
    if frame.isna().any().any():
        raise ValueError('Missing transition metadata; no rows were dropped')
    for c in required:
        frame[c] = frame[c].astype(str).str.strip()
        if frame[c].eq('').any():
            raise ValueError(f'Empty transition field: {c}')
    if not set(frame['coarse']).issubset(WITHIN_REFERENCE_EMOTIONS):
        raise ValueError('Target emotion is outside the CREMA-D reference inventory')
    frame = frame.loc[frame['transition'].isin(WITHIN_TRANSITIONS) & frame['coarse'].isin(WITHIN_COARSE)]
    keys = ['model', 'coarse', 'speaker', 'utterance', 'transition']
    if frame.empty or frame.duplicated(keys).any():
        raise ValueError('Empty transition table or duplicate pair identities')
    if set(frame['model']) != set(WITHIN_MODEL_NAMES):
        raise ValueError('Expected the original QWEN and FLAMINGO model keys')
    for model in WITHIN_MODEL_NAMES:
        if set(centroids[model]) - set(WITHIN_REFERENCE_EMOTIONS):
            raise ValueError('Supplied centroids include emotions outside the CREMA-D reference inventory')
    # The same MD anchor must be used for both target intensities.
    anchors = frame.groupby(['model', 'coarse', 'speaker', 'utterance'])['source_rep'].nunique()
    if anchors.gt(1).any():
        raise ValueError('HI/LO rows with one match key use different MD source files')
    expected = pd.MultiIndex.from_product([list(WITHIN_MODEL_NAMES), WITHIN_COARSE, WITHIN_TRANSITIONS])
    counts = frame.groupby(['model', 'coarse', 'transition']).size().reindex(expected, fill_value=0)
    if not counts.eq(42).all():
        raise ValueError(f'The original design has 42 pairs per cell; found {counts.to_dict()}')
    rows = []
    for model, group in frame.groupby('model', sort=True):
        cache = {}
        def embedding(path):
            if path not in cache:
                value = loader(path, WITHIN_LAYER)
                if value is None:
                    raise ValueError(f'Missing {WITHIN_LAYER} in {path}; no rows were dropped')
                cache[path] = within_normalized_vector(value)
            return cache[path]
        for index, (_, row) in enumerate(group.iterrows(), 1):
            centroid = within_normalized_vector(centroids[model][row['coarse']])
            source, target = embedding(row['source_rep']), embedding(row['target_rep'])
            if source.shape != centroid.shape or target.shape != centroid.shape:
                raise ValueError(f'Embedding/centroid dimension mismatch for {model}/{row["coarse"]}')
            # Explicit cosine, matching sklearn cosine_similarity in original cell 13.
            source_similarity = float(np.dot(source / np.linalg.norm(source), centroid / np.linalg.norm(centroid)))
            target_similarity = float(np.dot(target / np.linalg.norm(target), centroid / np.linalg.norm(centroid)))
            rows.append({'Reference_dataset': 'CREMA-D', 'Shift_dataset': 'CREMA-D',
                         'Model': WITHIN_MODEL_NAMES[model], 'Layer': WITHIN_LAYER,
                         'Coarse': row['coarse'], 'Transition': row['transition'],
                         'speaker': row['speaker'], 'utterance': row['utterance'],
                         'source_rep': row['source_rep'], 'target_rep': row['target_rep'],
                         'source_similarity': source_similarity, 'target_similarity': target_similarity,
                         'similarity_change': target_similarity - source_similarity})
            if index % 100 == 0 or index == len(group):
                print(f'Within-CREMA-D scores: {model}: {index}/{len(group)}')
    return pd.DataFrame(rows)


def within_prepare_scores(frame):
    """Validate provenance, paired IDs, cosine bounds and the subtraction identity."""
    frame = frame.copy()
    for column in ['Reference_dataset', 'Shift_dataset']:
        if column not in frame or not frame[column].eq('CREMA-D').all():
            raise ValueError(f'{column} must explicitly be CREMA-D; cross-dataset CSVs are invalid')
    required = ['source_similarity', 'target_similarity', 'similarity_change']
    if not set(required).issubset(frame.columns):
        raise ValueError('Need per-pair source_similarity, target_similarity and similarity_change; summary means are insufficient')
    for column in required:
        frame[column] = pd.to_numeric(frame[column], errors='raise').astype(float)
        if not np.isfinite(frame[column]).all():
            raise ValueError(f'Invalid {column}')
    if (frame[['source_similarity', 'target_similarity']].abs() > 1 + 1e-6).any().any():
        raise ValueError('Source/target cosine must be in [-1, 1]')
    if not np.allclose(frame['similarity_change'], frame['target_similarity'] - frame['source_similarity'], atol=1e-7, rtol=1e-6):
        raise ValueError('similarity_change must equal target_similarity minus source_similarity')
    # This checks identity completeness and score bounds; retain richer provenance.
    validate_within_pairs(frame)
    for column in WITHIN_GROUP_COLUMNS + ['Transition', 'speaker', 'utterance']:
        frame[column] = frame[column].astype(str).str.strip()
    expected = {(m, WITHIN_LAYER, e) for m in WITHIN_MODEL_NAMES.values() for e in WITHIN_COARSE}
    actual = set(frame[['Model', 'Layer', 'Coarse']].itertuples(index=False, name=None))
    if actual != expected:
        raise ValueError(f'Expected all 10 model/layer/emotion groups for the planned 20 tests. Missing={expected-actual}; unexpected={actual-expected}')
    key = WITHIN_GROUP_COLUMNS + ['speaker', 'utterance']
    spread = frame.groupby(key)['source_similarity'].agg(['min', 'max'])
    if (spread['max'] - spread['min'] > 1e-6).any():
        raise ValueError('Matched HI/LO rows have inconsistent MD source similarity')
    if 'source_rep' in frame and frame.groupby(key)['source_rep'].nunique().gt(1).any():
        raise ValueError('Matched HI/LO rows have inconsistent MD source paths')
    frame['Target_emotion'] = frame['Coarse']
    frame['Comparison'] = frame['Transition'].map({'MD_TO_HI': 'HI_minus_MD', 'MD_TO_LO': 'MD_minus_LO'})
    frame['expected_direction_score'] = frame['similarity_change'] * np.where(frame.Transition.eq('MD_TO_HI'), 1, -1)
    return frame.sort_values(key + ['Transition'], kind='stable').reset_index(drop=True)


def within_cremad_inference(frame, *, alpha=WITHIN_ALPHA, **resampling):
    """Twenty primary tests: HI-MD and MD-LO, each expected to be positive.

    Each score is already a source/target matched difference for one recording
    pair and its annotated emotion centroid. No HI/LO intersection is required.
    Two-sided sign-flip tests on cluster sums are retained; the effect sign determines
    support for or contradiction of the expected direction after Holm correction.
    """
    if not 0 < alpha < 1:
        raise ValueError('alpha must be between zero and one')
    frame = within_prepare_scores(frame)
    results, speaker_tables = [], []
    for keys, group in frame.groupby(WITHIN_GROUP_COLUMNS, sort=True):
        for transition in WITHIN_TRANSITIONS:
            available = group.loc[group.Transition.eq(transition)]
            orientation = 1.0 if transition == 'MD_TO_HI' else -1.0
            comparison = 'HI_minus_MD' if orientation == 1 else 'MD_minus_LO'
            values = available['similarity_change'].to_numpy() * orientation
            meta = dict(zip(WITHIN_GROUP_COLUMNS, keys))
            meta.update({'Test': 'intensity_vs_MD', 'Transition': transition,
                         'Comparison': comparison, 'Target_emotion': keys[-1],
                         'expected_sign': 'positive',
                         'expected_movement': 'toward_centroid' if orientation == 1 else 'away_from_centroid',
                         'analysis_population': 'all_available_MD_target_pairs',
                         'mean_target_minus_MD': float(available.similarity_change.mean()) if len(available) else np.nan})
            n_speakers = available.speaker.nunique()
            if n_speakers < 2:
                results.append({**meta,
                    'status': 'not_tested_no_rows' if available.empty else 'not_tested_fewer_than_2_speakers',
                    'estimate': float(values.mean()) if len(values) else np.nan,
                    'CI95_low': np.nan, 'CI95_high': np.nan, 'p_raw': np.nan,
                    'N_pairs': len(available), 'N_speakers': n_speakers,
                    'test_method': 'not_tested', 'alternative': 'two-sided', 'null_value': 0.0,
                    'sign_patterns_evaluated': 0, 'bootstrap_draws': 0,
                    'fewer_than_10_speakers': bool(n_speakers < 10)})
                continue
            result, speakers = within_speaker_cluster_inference(
                values, available.speaker.to_numpy(), tuple(keys) + ('intensity_vs_MD', comparison), **resampling)
            results.append({**meta, 'status': 'tested', **result})
            for column, value in meta.items():
                speakers[column] = value
            speaker_tables.append(speakers)
    result_rows = results
    results = pd.DataFrame(results)
    for column in ['sign_flip_seed', 'bootstrap_seed']:
        results[column] = pd.array([row.get(column, pd.NA) for row in result_rows], dtype='UInt64')
    assert len(results) == 20
    tested = results.status.eq('tested')
    results['p_holm'] = within_holm_adjust(results.p_raw.fillna(1).to_numpy())
    results.loc[~tested, 'p_holm'] = np.nan
    results['holm_family_size'] = 20
    results['N_tests_performed'] = int(tested.sum())
    results['alpha'] = alpha
    results['reject_holm'] = pd.array(results.p_holm.le(alpha), dtype='boolean')
    results.loc[~tested, 'reject_holm'] = pd.NA
    significant = results.reject_holm.fillna(False).to_numpy(dtype=bool)
    results['conclusion'] = np.select(
        [~tested, significant & results.estimate.gt(0), significant & results.estimate.lt(0)],
        ['not_tested', 'supports_expected_direction', 'opposes_expected_direction'],
        default='no_significant_evidence')
    results['estimated_movement_relative_to_MD'] = np.select(
        [results.mean_target_minus_MD.gt(0), results.mean_target_minus_MD.lt(0), results.mean_target_minus_MD.eq(0)],
        ['toward_centroid', 'away_from_centroid', 'no_mean_change'], default='unavailable')
    results = results.sort_values(WITHIN_GROUP_COLUMNS + ['Comparison'], kind='stable').reset_index(drop=True)
    speaker_table = pd.concat(speaker_tables, ignore_index=True) if speaker_tables else pd.DataFrame()
    return results, speaker_table


def within_category_coverage(frame, centroids=None):
    """Show the seven-category taxonomy separately from actual data coverage."""
    rows = []
    for alias, model in WITHIN_MODEL_NAMES.items():
        group = frame.loc[frame.Model.eq(model)]
        for emotion in WITHIN_COARSE_TAXONOMY:
            emotion_rows = group.loc[group.Coarse.eq(emotion)]
            if emotion == 'SUR':
                status = 'not_in_CREMA_D_reference_inventory'
            elif emotion not in WITHIN_COARSE:
                status = 'no_MD_HI_LO_pairs_in_this_evaluation'
            else:
                status = 'included_in_primary_tests'
            rows.append({'Model': model, 'Layer': WITHIN_LAYER, 'Target_emotion': emotion,
                         'category_status': status,
                         'centroid_verified_in_memory': pd.NA if centroids is None else emotion in centroids[alias],
                         'N_HI_MD_pairs': int(emotion_rows.Transition.eq('MD_TO_HI').sum()),
                         'N_MD_LO_pairs': int(emotion_rows.Transition.eq('MD_TO_LO').sum())})
    return pd.DataFrame(rows)


In [ ]:
_within_inventory = pd.DataFrame()
_within_inputs = []
_within_centroids = None
if WITHIN_PAIRS_CSV is not None:
    _within_csv = Path(WITHIN_PAIRS_CSV).expanduser()
    _within_input = pd.read_csv(_within_csv, dtype={c: str for c in WITHIN_GROUP_COLUMNS + ['speaker', 'utterance', 'Transition']})
    _within_source = str(_within_csv.resolve())
    _within_inputs.append({'path': _within_source, 'sha256': hashlib.sha256(_within_csv.read_bytes()).hexdigest()})
elif 'within_cremad_pair_scores' in globals():
    _within_input = within_cremad_pair_scores.copy()
    _within_source = 'in-memory within_cremad_pair_scores with explicit pair IDs'
else:
    if 'transition_df' in globals():
        _within_transitions = transition_df.copy()
        _within_inputs.append({'source': 'in-memory transition_df from original geometry.ipynb'})
    else:
        _missing = [p for p in WITHIN_TRANSITION_CSVS.values() if not Path(p).is_file()]
        if _missing:
            raise FileNotFoundError('The raw data are on Google Drive. In Colab, mount the original Drive, then run this section; or set WITHIN_PAIRS_CSV to within_cremad_pair_scores.csv. Missing: ' + repr(_missing))
        _within_transitions, _within_inputs = within_load_transition_manifest()
    if 'model_centroids' in globals():
        if globals().get('TARGET_LAYER') != WITHIN_LAYER:
            raise ValueError('Cannot verify that the in-memory model_centroids use lm_2')
        _within_centroids = model_centroids
        _within_inputs.append({'source': 'in-memory model_centroids from original cell 12', 'reference_uncertainty': 'fixed'})
    else:
        _within_centroids, _within_inventory, _within_reference_inputs = within_load_reference_centroids()
        _within_inputs.extend(_within_reference_inputs)
    _within_input = within_scores_from_transitions(_within_transitions, _within_centroids)
    _within_source = 'recomputed per-pair similarities using the original within-CREMA-D metric'

within_cremad_pair_scores = within_prepare_scores(_within_input)
within_cremad_statistics, within_cremad_speaker_scores = within_cremad_inference(within_cremad_pair_scores)
within_cremad_category_coverage = within_category_coverage(within_cremad_pair_scores, _within_centroids)
assert len(within_cremad_statistics) == 20

if 'RUN_DIR' in globals():
    _within_parent = Path(RUN_DIR)
elif WITHIN_PAIRS_CSV is not None:
    _within_parent = Path(WITHIN_PAIRS_CSV).expanduser().resolve().parent
elif Path('/content/drive/MyDrive').is_dir():
    _within_parent = Path('/content/drive/MyDrive/subprobe/geometry_unified')
else:
    _within_parent = Path.cwd() / 'geometry_unified'
_within_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
WITHIN_STATS_DIR = _within_parent / f'within_cremad_statistics_{_within_stamp}_{uuid.uuid4().hex[:8]}'
WITHIN_STATS_DIR.mkdir(parents=True, exist_ok=False)
for filename, table in {
    'within_cremad_pair_scores.csv': within_cremad_pair_scores,
    'within_cremad_statistics.csv': within_cremad_statistics,
    'within_cremad_speaker_scores.csv': within_cremad_speaker_scores,
    'within_cremad_category_coverage.csv': within_cremad_category_coverage,
}.items():
    table.to_csv(WITHIN_STATS_DIR / filename, index=False)
if not _within_inventory.empty:
    _within_inventory.to_csv(WITHIN_STATS_DIR / 'within_cremad_reference_inventory.csv', index=False)
if _within_centroids is not None:
    np.savez_compressed(WITHIN_STATS_DIR / 'within_cremad_reference_centroids.npz',
                        **{model + '__' + emotion: vector for model, group in _within_centroids.items() for emotion, vector in group.items()})

within_cremad_manifest = {
    'created_utc': _within_stamp, 'input_source': _within_source, 'input_metadata': _within_inputs,
    'exported_pair_sha256': hashlib.sha256((WITHIN_STATS_DIR / 'within_cremad_pair_scores.csv').read_bytes()).hexdigest(),
    'metric': 'cos(target, fixed coarse centroid) - cos(MD, fixed coarse centroid)',
    'reference_construction': 'expected original geometry.ipynb construction: float32 normalized reference vectors, arithmetic mean by coarse emotion, normalize centroid; not independently verified for precomputed CSV inputs',
    'conditioning': 'fixed CREMA-D reference centroids; no resampling of reference estimation uncertainty; reference/evaluation speaker independence not verified',
    'statistic': 'pair-weighted mean HI-minus-MD or MD-minus-LO cosine similarity change; both expected positive',
    'raw_score': 'similarity_change remains target-minus-MD; low-condition score is negated for the MD-minus-LO test',
    'target_selection': 'annotated coarse emotion, fixed for both members of every pair; not model prediction or nearest centroid',
    'resampling_unit': 'speaker; all observations from a speaker move together',
    'nulls': ['mean HI-minus-MD cosine similarity = 0', 'mean MD-minus-LO cosine similarity = 0'],
    'alternative': 'two-sided, absolute mean',
    'assumptions': 'independent speaker clusters and sign-symmetric cluster score vectors under H0; zero mean alone is insufficient',
    'n_sign_flips': WITHIN_N_SIGN_FLIPS, 'exact_if_speakers_at_most': WITHIN_MAX_EXACT_SPEAKERS,
    'p_value': 'exact extreme/all patterns; Monte Carlo (extreme+1)/(draws+1)',
    'n_bootstrap': WITHIN_N_BOOTSTRAP, 'confidence_intervals': 'pointwise 95% percentile whole-speaker bootstrap',
    'seed': WITHIN_STATS_SEED, 'alpha': WITHIN_ALPHA, 'holm_family_size': 20,
    'expected_direction_decision': 'significant positive oriented estimate supports expectation; significant negative opposes; nonsignificant is inconclusive',
    'taxonomy': WITHIN_COARSE_TAXONOMY, 'CREMA_D_reference_inventory': WITHIN_REFERENCE_EMOTIONS,
    'evaluated_emotions': WITHIN_COARSE,
    'multiple_testing_scope': 'this within-CREMA-D family only, not jointly with the previous cross-dataset tests',
    'matching': 'MD and its HI or LO counterpart have the same speaker, utterance and annotated emotion; no HI/LO intersection required',
    'unavailable_tests': 'not tested with fewer than 2 speakers; p=1 only internally for Holm; exported p-values stay missing',
    'n_tests_performed': int(within_cremad_statistics['status'].eq('tested').sum()),
    'python': __import__('platform').python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
}
(WITHIN_STATS_DIR / 'within_cremad_statistics_manifest.json').write_text(json.dumps(within_cremad_manifest, indent=2), encoding='utf-8')
try:
    from IPython.display import display as _within_display
except ImportError:
    _within_display = print
_within_columns = ['Model', 'Layer', 'Target_emotion', 'Comparison', 'status', 'estimate', 'CI95_low', 'CI95_high',
                   'N_pairs', 'N_speakers', 'p_raw', 'p_holm', 'reject_holm', 'conclusion']
print('Primary comparisons: positive HI-minus-MD and MD-minus-LO estimates support the expected movement.')
_within_display(within_cremad_statistics[_within_columns].round(6))
print('Seven-category taxonomy versus actual CREMA-D coverage:')
_within_display(within_cremad_category_coverage)
print('20 primary within-CREMA-D tests; no HI-versus-LO test; pointwise bootstrap CIs; Holm alpha=0.05.')
print('Unavailable tests are NOT nonsignificant findings. Inspect status and N_speakers.')
print('Results CSV:', WITHIN_STATS_DIR / 'within_cremad_statistics.csv')
print('Reusable pair CSV:', WITHIN_STATS_DIR / 'within_cremad_pair_scores.csv')
